# Modelagem — Whirlpool Thermal Predictor

Este notebook é o quarto da pipeline de análise, responsável por treinar e avaliar modelos preditivos de temperatura para as diferentes áreas térmicas do fogão Whirlpool.

O ponto de partida são três insumos: os datasets tratados pelos notebooks anteriores (`dataset_*_13cat.csv`), a tabela de limites de segurança por componente (`limites_ativos.csv`) e a partição gerada pelo notebook `split_treino_teste.ipynb`. A partição não é sorteada aqui — ela é lida de um arquivo JSON gerado previamente, garantindo que todos os modelos sejam treinados e avaliados exatamente sobre o mesmo conjunto de ensaios. Sem essa fixação, a variância do sorteio introduz desvios de até 2,25°C nas métricas — maior que o ruído experimental de algumas áreas termicas — o que inviabilizaria comparações justas entre modelos.

## Como este caderno se organiza

São vinte e quatro seções, e a entrega está na seção 23. O mapa abaixo evita que se leia da primeira à última para descobrir isso.

| Seções | O que contêm | Vale ler para |
|---|---|---|
| 1 a 6 | Configuração, leitura dos conjuntos, partição fixa e definição das métricas | Entender o protocolo antes de qualquer número |
| 7 a 14 | Modelos avaliados um a um, do modelo linear de referência ao gradiente, e a decisão contra o limite de segurança | Saber o que já foi tentado e descartado |
| 15 a 18 | Alternativas descartadas, valores extremos, comparação entre as duas versões da base e o caso das placas de alumínio | Não repetir caminhos já fechados |
| 19 e 20 | Formulação de referência, com floresta em todas as regiões, na validação e no teste reservado | Ter o ponto de comparação dos ganhos posteriores |
| 21 e 22 | Seleção de codificação, estimador e escala do alvo por região, e a variabilidade da estimativa | Entender por que a grade larga de seleção foi rejeitada |
| 23 | Formulação entregue da quarta sprint | Saber o que de fato é entregue |
| 24 | Diagnóstico do coeficiente de determinação do Oven HMI | Entender por que o R² da região é baixo |

A formulação entregue escolhe, por região, o estimador entre a floresta aleatória e a média entre floresta e gradiente, e o tratamento das perdas declaradas entre os treze indicadores originais e quatro níveis mutuamente exclusivos, sobre a codificação de treze categorias com o motivo do ensaio agrupado em seis níveis, dá mais peso no treino às leituras próximas do limite de cada componente e, em Main Vent Sys, usa a média entre Extra Trees e gradiente. Ela mede de 5,96 a 6,23 graus de erro na validação cruzada, conforme o método de aninhamento, contra 6,107 da formulação que escolhia apenas o estimador, medida na mesma execução, 6,47 da formulação de referência (seção 19) e 6,565 do modelo da Sprint 3, medido na base anterior. No conjunto de teste reservado mede 5,154 graus, contra 5,17 da formulação que escolhia apenas o estimador, 5,15 da formulação de referência (seção 20) e 4,37 do modelo da Sprint 3, este também medido na base anterior e por isso não diretamente comparável.

O segundo eixo da escolha veio de uma auditoria da qualidade do dado bruto, e não de algoritmo novo, o que repete o padrão de toda a sprint: os ganhos vieram do dado e da sua codificação. A auditoria está descrita na seção 23 e as suas demais constatações, entre elas a confusão perfeita entre o tipo de ensaio pyro e a ausência de perdas, no relatório do projeto.

Três resultados negativos estão registrados aqui e devem ser lidos antes de qualquer proposta de ampliar a busca. A seleção sobre uma grade de doze combinações rende menos do que sobre duas. As colunas de ordem de chaveamento pioram o resultado. E o conjunto de teste reservado aponta em sentido contrário ao da validação, por razões que a seção 20 e o relatório comparativo discutem.

## 1. Configuração

No Google Colab, suba os sete arquivos `dataset_*_13cat.csv`, o `limites_ativos.csv` e o `particao_treino_teste.json` pela barra lateral (ícone de pasta) e rode a célula abaixo.

No repositório local, os arquivos serão encontrados automaticamente pela função `resolve()`.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os, glob, json
import warnings

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

import pandas as pd
import numpy as np

# Versao adotada: 13 categorias originais da coluna de perdas, com 27 colunas
# de entrada. A comparacao com a versao de perfil de perdas, sob este mesmo
# protocolo e a mesma particao, resultou em 7,543 contra 7,560 graus de erro
# absoluto medio, diferenca vinte vezes inferior ao desvio de 0,35 grau da
# propria estimativa; as duas versoes empatam sob essa medida.
# O criterio de desempate anterior, registrado na Secao 4.3.1, adotava a versao
# de menor dimensionalidade. A adocao das 13 categorias e uma decisao posterior,
# apoiada em dois resultados independentes: a selecao aninhada da Secao 21 e a
# grade exaustiva do caderno de pipeline escolheram essa codificacao para as
# regioes de bobinas e de ventilacao, que sao as de maior erro.
SUFIXO = "_13cat"
PASTA_DADOS    = "colabs_S4"
# A quarta sprint distribui apenas a codificacao de treze categorias. A versao
# de perfil de perdas permanece na pasta da terceira sprint e e lida de la
# somente pelas secoes cujo objeto e comparar as duas codificacoes.
# Sem esta leitura explicita a codificacao chamada perfil passava a ser montada
# a partir do conjunto carregado com SUFIXO, isto e, coincidia com a de treze
# categorias, e a grade da Secao 21 comparava duas copias da mesma tabela. O
# defeito era silencioso e produziu um ganho falso de -0,246 antes da correcao.
PASTA_DADOS_PERFIL = "colabs_S4"   # versao perfil gerada no tratamento_datasets_13cat, secao 10
PASTA_PARTICAO = "colabs_S3/preparacao_testes"

BASES = [
    ".", "..", os.path.join("..", ".."), "/content",
    "notebooks", os.path.join("..", "notebooks"),
    os.path.join("notebooks", PASTA_DADOS),
    os.path.join("notebooks", PASTA_PARTICAO),
]

def resolve(pasta, arquivo):
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas)
    )

print("Diretorio atual:", os.path.abspath("."))
print("Versao de tratamento das perdas:", SUFIXO)

## 2. Leitura dos datasets tratados

Os arquivos `dataset_*_13cat.csv` são os produtos finais do pré-processamento. Cada um representa uma área térmica do fogão, já com One-Hot Encoding aplicado. A coluna `run_idx`, quando presente, é usada para indexar a partição; `indices_da_dobra()` trata ambos os casos — com e sem ela.

In [ ]:
PREFIXOS_FEATURES = [
    "blower_direction", "housing", "board_position", "run_case",
    "losses", "perfil_perdas", "deflector_hole", "material_coil_support",
    # Familia do motivo do ensaio e ordem de chaveamento, montadas na preparacao.
    # Sem estes prefixos declarados, as colunas seriam tratadas como colunas-alvo,
    # e o erro so apareceria adiante, ao calcular o erro percentual sobre valores
    # que nao sao temperatura. Os nomes antigos permanecem para que o caderno
    # continue a ler os conjuntos da terceira sprint.
    "cfg", "perfil_perdas_ordem",
    "reason", "loss_profile_order",
]
ORDEM_ACIONAMENTO = ("perfil_perdas_ordem_", "loss_profile_order_")  # nome antigo, da S3, e novo, da S4
COLUNAS_AUXILIARES = ["run_idx"]

arquivos = sorted(glob.glob(os.path.join(
    os.path.dirname(resolve(PASTA_DADOS, f"dataset_mainback{SUFIXO}.csv")),
    f"dataset_*{SUFIXO}.csv",
)))
print(f"Arquivos encontrados: {len(arquivos)}")
for a in arquivos:
    print(" ", os.path.basename(a))

In [ ]:
datasets = {}
for arq in arquivos:
    nome = os.path.basename(arq).replace("dataset_", "").replace(SUFIXO + ".csv", "")
    datasets[nome] = pd.read_csv(arq)
print("Datasets carregados:", len(datasets))

In [ ]:
limites = pd.read_csv(resolve(PASTA_DADOS, "limites_ativos.csv")).set_index("tc_name")
print("Tabela de limites carregada:", len(limites), "componentes")

with open(resolve(PASTA_PARTICAO, "particao_treino_teste.json"), encoding="utf-8") as f:
    particao = json.load(f)

IDX_TREINO = particao["run_idx_treino"]
IDX_TESTE  = particao["run_idx_teste"]
DOBRA  = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
DOBRAS = sorted({d for d in DOBRA.values() if d is not None})

print(f"Ensaios de treino: {len(IDX_TREINO)}  |  Ensaios de teste: {len(IDX_TESTE)}")
print(f"Dobras de validacao cruzada: {DOBRAS}")


def limite_de(componente):
    """Limite de seguranca do componente, ou None quando nao houver."""
    valor = limites["limite_num"].get(componente)
    return float(valor) if valor is not None and pd.notna(valor) else None


print("Componentes com limite tabelado:",
      int(limites["limite_num"].notna().sum()))

## 3. Conferencia geral

In [ ]:
N_ENSAIOS_ESPERADO = particao["n_ensaios"]
print("{:<16} {:>8} {:>9} {:>6} {:>6}  {}".format("Dataset","Ensaios","Entradas","Alvos","Total","Status"))
print("-" * 60)
ok_geral = True
for nome, df in sorted(datasets.items()):
    cols_entrada = [c for c in df.columns if any(c.startswith(p+"_") for p in PREFIXOS_FEATURES)]
    cols_alvo    = [c for c in df.columns if c not in cols_entrada + COLUNAS_AUXILIARES]
    erros = []
    if len(df) != N_ENSAIOS_ESPERADO: erros.append(f"{len(df)} linhas (esperado {N_ENSAIOS_ESPERADO})")
    if not cols_entrada: erros.append("nenhuma feature")
    if not cols_alvo:    erros.append("nenhum alvo")
    # Trava contra prefixo nao declarado: alvo e temperatura absoluta, sempre
    # positiva. Uma coluna de entrada que escape da lista de prefixos cai entre os
    # alvos e traz zeros, e e aqui que isso deve aparecer, e nao trinta celulas
    # adiante, dentro do calculo de erro percentual.
    nao_positivos = [c for c in cols_alvo
                     if (df[c].dropna() <= 0).any()]
    if nao_positivos:
        erros.append(f"alvo com valor nao positivo, prefixo nao declarado? {nao_positivos[:4]}")
    status = "OK" if not erros else "ERRO: " + " | ".join(erros)
    if erros: ok_geral = False
    tag = "(+run_idx)" if "run_idx" in df.columns else ""
    print("{:<16} {:>8} {:>9} {:>6} {:>6}  {} {}".format(
        nome, len(df), len(cols_entrada), len(cols_alvo), len(df.columns), status, tag))
print()
print("Verificacao:", "PASSOU" if ok_geral else "FALHOU")

## 4. Separacao entre features e colunas-alvo

In [ ]:
def separar(df, prefixos=PREFIXOS_FEATURES):
    """
    Devolve (X, cols_alvo). Colunas em COLUNAS_AUXILIARES (run_idx)
    sao excluidas de features e alvos -- usadas apenas para indexar a particao.
    """
    cols_entrada = [c for c in df.columns if any(c.startswith(p+"_") for p in prefixos)]
    cols_alvo    = [c for c in df.columns if c not in cols_entrada + COLUNAS_AUXILIARES]
    return df[cols_entrada].copy(), cols_alvo

print(f"{'Dataset':<16} {'Features':>8} {'Alvos':>6}")
print("-" * 34)
for nome, df in sorted(datasets.items()):
    X, cols_alvo = separar(df)
    print(f"{nome:<16} {X.shape[1]:>8} {len(cols_alvo):>6}")

## 5. Partição e índices por dobra

A função `indices_da_dobra()` usa a partição lida do JSON para retornar os índices de treino e validação de uma dobra, aplicando a máscara de nulos. Os ensaios do conjunto de teste permanecem completamente fora durante toda a fase de validação cruzada.

In [ ]:
def indices_da_dobra(df, dobra, y_col):
    """
    Retorna (idx_treino, idx_validacao) para uma dobra especifica.
    Usa run_idx quando disponivel; caso contrario, indice posicional.
    """
    run = df["run_idx"].to_numpy() if "run_idx" in df.columns else np.arange(len(df))
    medido    = y_col.notna().to_numpy()
    no_treino = np.array([r in DOBRA and DOBRA[r] is not None for r in run])
    idx_val = [i for i in range(len(df)) if medido[i] and no_treino[i] and DOBRA[run[i]] == dobra]
    idx_tr  = [i for i in range(len(df)) if medido[i] and no_treino[i] and DOBRA[run[i]] != dobra]
    return idx_tr, idx_val


df_ex = datasets["mainback"]
X_ex, cols_ex = separar(df_ex)
coluna_ex = next(c for c in cols_ex if df_ex[c].notna().any())
y_ex = df_ex[coluna_ex]
print(f"Dataset: mainback  |  Coluna-alvo: {coluna_ex}")
print(f"Ensaios com medicao: {y_ex.notna().sum()} de {len(y_ex)}")
print()
print(f"{'Dobra':>6} {'Treino':>8} {'Validacao':>10}")
print("-" * 28)
for d in DOBRAS:
    tr, val = indices_da_dobra(df_ex, d, y_ex)
    print(f"{d:>6} {len(tr):>8} {len(val):>10}")

## 6. Métricas de avaliação

As funções abaixo são a definição canônica das métricas do projeto, copiada sem alteração da Seção 1 do notebook metricas.ipynb, onde constam a fórmula de cada uma, a justificativa da escolha e a bateria de casos de teste que as verifica.

A definição fica em um lugar só por dois motivos. O primeiro é de correção: antes desta consolidação, cada família de modelo recalculava as mesmas três métricas com código próprio, de modo que uma correção precisaria ser aplicada quatro vezes e uma divergência entre elas passaria despercebida. O segundo é de extensão: o acréscimo do erro percentual e das métricas de decisão contra o limite de segurança passa a alcançar todos os modelos de uma vez, sem alteração em ponto de chamada algum, uma vez que a avaliação devolve um dicionário.

Duas particularidades da implementação merecem registro. O coeficiente de determinação é devolvido como ausente quando o alvo assume um único valor, situação em que ele é indefinido e em que a implementação do scikit-learn devolve zero, valor que se confunde com um resultado legítimo. E a raiz do erro quadrático é calculada de forma explícita, uma vez que o parâmetro que a produzia foi removido do scikit-learn na versão 1.9.

In [ ]:
MARGEM_KPI = 0.10   # margem de erro acordada no TAPI


def _valida(y_real, y_previsto):
    """Converte para vetores e recusa entradas que produziriam metrica sem sentido.

    A conversao para numpy e necessaria porque a subtracao entre duas Series do pandas
    alinha pelos indices, e nao pela posicao. Uma Series com indice original e outra
    com indice reiniciado produziriam diferencas entre pares errados, em silencio.

    O alvo precisa ser estritamente positivo. A restricao e legitima neste projeto,
    pois trata-se de temperatura absoluta em graus Celsius, cujo minimo na base e de
    27,69 graus. Ela existe para transformar em erro visivel o engano de passar valores
    padronizados, caso em que o alvo se aproximaria de zero e o erro percentual
    explodiria em vez de acusar o problema.
    """
    y_real = np.asarray(y_real, dtype=float).ravel()
    y_previsto = np.asarray(y_previsto, dtype=float).ravel()

    if y_real.shape != y_previsto.shape:
        raise ValueError(f"dimensoes diferentes: {y_real.shape} e {y_previsto.shape}")
    if y_real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(y_real).any():
        raise ValueError("ha valores ausentes em y_real; a mascara deve ser aplicada antes")
    if np.isnan(y_previsto).any():
        raise ValueError("ha valores ausentes em y_previsto")
    if (y_real <= 0).any():
        raise ValueError(
            "y_real contem valor nao positivo; o erro percentual exige temperatura "
            "absoluta em graus Celsius, e nao valores padronizados")

    return y_real, y_previsto


def avaliar(y_real, y_previsto, limite=None):
    """Metricas de erro de um componente, com o alvo em graus Celsius.

    Sempre devolvidas:

    n            numero de observacoes que sustentam as metricas
    MAE          erro absoluto medio, na unidade do alvo
    RMSE         raiz do erro quadratico medio, na mesma unidade
    R2           fracao da variancia do alvo explicada pelo modelo
    vies         erro medio com sinal, positivo quando o modelo superestima
    RMSE/MAE     razao entre as duas, indicando concentracao dos erros
    MAPE_%       erro percentual absoluto medio
    WAPE_%       erro absoluto total dividido pela soma dos alvos
    dentro_10_%  proporcao de predicoes dentro da margem de 10 por cento do TAPI

    Devolvidas apenas quando um limite de seguranca e informado:

    limite_C          o proprio limite, para conferencia
    n_reprova_real    ensaios em que o componente de fato atinge o limite
    acerto_decisao_%  concordancia entre a decisao real e a prevista
    falso_aprovado    ensaios reprovados na realidade e aprovados pelo modelo
    falso_reprovado   ensaios aprovados na realidade e reprovados pelo modelo

    O R2 sai ausente quando o alvo assume um unico valor, situacao em que ele e
    indefinido e em que a implementacao do scikit-learn devolve zero, valor que se
    confunde com um resultado legitimo.
    """
    y_real, y_previsto = _valida(y_real, y_previsto)

    erro = y_previsto - y_real
    mae = float(np.mean(np.abs(erro)))

    # A raiz e calculada explicitamente: o parametro squared do mean_squared_error foi
    # removido do scikit-learn na versao 1.9, e o root_mean_squared_error so existe a
    # partir da 1.4. O calculo direto funciona em qualquer versao.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(y_real).size < 2:
        r2 = float("nan")
    else:
        ss_res = float(np.sum(erro ** 2))
        ss_tot = float(np.sum((y_real - np.mean(y_real)) ** 2))
        r2 = 1.0 - ss_res / ss_tot

    relativo = np.abs(erro) / y_real

    resultado = {
        "n": int(y_real.size),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "vies": float(np.mean(erro)),
        "RMSE/MAE": (rmse / mae) if mae > 0 else float("nan"),
        "MAPE_%": 100.0 * float(np.mean(relativo)),
        "WAPE_%": 100.0 * float(np.sum(np.abs(erro)) / np.sum(y_real)),
        "dentro_10_%": 100.0 * float(np.mean(relativo <= MARGEM_KPI)),
    }

    if limite is not None and np.isfinite(limite):
        reprova_real = y_real >= limite
        reprova_prevista = y_previsto >= limite
        resultado.update({
            "limite_C": float(limite),
            "n_reprova_real": int(reprova_real.sum()),
            "acerto_decisao_%": 100.0 * float(np.mean(reprova_real == reprova_prevista)),
            "falso_aprovado": int(np.sum(reprova_real & ~reprova_prevista)),
            "falso_reprovado": int(np.sum(~reprova_real & reprova_prevista)),
        })

    return resultado


def consolidar(tabela, grupo="dataset",
               metricas=("MAE", "RMSE", "R2", "vies", "MAPE_%", "WAPE_%", "dentro_10_%")):
    """Consolida metricas por componente em metricas por area.

    A macro-media atribui o mesmo peso a cada componente e responde se o modelo funciona
    bem em todas as partes do produto. A micro-media pondera pelo numero de observacoes
    e responde qual o erro tipico de uma leitura qualquer.

    O R2 nao e comparavel entre areas, pois seu denominador e a variancia do alvo de
    cada area. Ele indica se o modelo supera a media dentro da propria area.
    """
    if grupo not in tabela.columns:
        raise ValueError(f"coluna de agrupamento ausente: {grupo}")
    if "n" not in tabela.columns:
        raise ValueError("a tabela precisa da coluna n para a micro-media")

    metricas = [m for m in metricas if m in tabela.columns]
    linhas = []

    for nome, bloco in tabela.groupby(grupo):
        linha = {grupo: nome, "componentes": len(bloco), "n_total": int(bloco["n"].sum())}
        for m in metricas:
            valores = bloco[m]
            validos = valores.notna()
            linha[f"{m}_macro"] = (float(valores[validos].mean())
                                   if validos.any() else float("nan"))
            if validos.any() and bloco.loc[validos, "n"].sum() > 0:
                linha[f"{m}_micro"] = float(
                    np.average(valores[validos], weights=bloco.loc[validos, "n"]))
            else:
                linha[f"{m}_micro"] = float("nan")

        # As metricas de decisao sao contagens, e por isso somadas em vez de promediadas
        for contagem in ("n_reprova_real", "falso_aprovado", "falso_reprovado"):
            if contagem in bloco.columns:
                linha[contagem] = int(bloco[contagem].fillna(0).sum())
        linhas.append(linha)

    return pd.DataFrame(linhas).sort_values(grupo).reset_index(drop=True)


print("Metricas definidas: avaliar, consolidar")

## 7. Modelo base — Ridge Regression

Ridge (regularização L2) é usado como baseline por lidar bem com a multicolinearidade entre as features binárias geradas pelo One-Hot Encoding e por ter apenas um hiperparâmetro a ajustar (`alpha`), selecionado por Generalized Cross-Validation dentro de cada fold.

As métricas são calculadas uma única vez sobre todas as predições acumuladas dos folds, em vez de serem calculadas separadamente por fold e depois mediadas. Esse protocolo evita a instabilidade do R² quando os folds têm poucos pontos — situação comum aqui, dado o tamanho dos datasets.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

ALPHAS_RIDGE  = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
MIN_TREINO    = 8
MIN_PREDICOES = 5

resultados_ridge = []
for nome, df in sorted(datasets.items()):
    X, cols_alvo = separar(df)
    for col in cols_alvo:
        y_col = df[col]
        if y_col.notna().sum() < MIN_PREDICOES: continue
        reais, preditos, alphas_usados = [], [], []
        for d in DOBRAS:
            idx_tr, idx_val = indices_da_dobra(df, d, y_col)
            if len(idx_tr) < MIN_TREINO or not idx_val: continue
            scaler = StandardScaler()
            X_tr_sc = scaler.fit_transform(X.iloc[idx_tr])
            X_val_sc = scaler.transform(X.iloc[idx_val])
            model = RidgeCV(alphas=ALPHAS_RIDGE)
            model.fit(X_tr_sc, y_col.iloc[idx_tr])
            reais.extend(y_col.iloc[idx_val].tolist())
            preditos.extend(model.predict(X_val_sc).tolist())
            alphas_usados.append(model.alpha_)
        if len(reais) < MIN_PREDICOES: continue
        reais = np.array(reais); preditos = np.array(preditos)
        resultados_ridge.append({
            "dataset": nome, "componente": col,
            **avaliar(reais, preditos, limite=limite_de(col)),
            "alpha_modal": float(max(set(alphas_usados), key=alphas_usados.count)),
        })

df_ridge = pd.DataFrame(resultados_ridge)
print(f"Componentes avaliados: {len(df_ridge)}")
print()
print(df_ridge.groupby("dataset")[["MAE","RMSE","R2"]].mean().round(3).to_string())

### Piso de ruído da base, medido sobre a base atual

O piso de ruído é a dispersão dos alvos entre ensaios que o modelo não tem como distinguir, isto é, ensaios cujo vetor de entrada é idêntico. Nenhum modelo pode errar menos do que isso, porque para esses ensaios ele é obrigado a emitir a mesma previsão enquanto os valores medidos discordam.

O cálculo é feito aqui, a partir dos próprios dados, e não mais lido de constantes. A razão é que os valores anteriormente registrados foram medidos sobre uma versão anterior da base e sobre uma chave de configuração de sete variáveis, e deixaram de valer por duas vias. A primeira é que a extração dos atributos físicos do motivo do ensaio elevou a chave de sete para doze variáveis, separando ensaios que antes eram contabilizados como repetições e cuja discordância era atribuída a ruído quando decorria de diferença física declarada. A segunda é que os valores registrados não são reproduzíveis nem sobre a base de que teriam vindo, o que impede sua auditoria.

Dois estimadores são reportados. A média dos desvios-padrão por grupo mantém a comparabilidade com o procedimento anterior, mas subestima a dispersão em grupos pequenos. A raiz da média das variâncias, ponderada pelos graus de liberdade, é o estimador correto da dispersão comum e constitui o valor a ser preferido. A distância entre os dois indica concentração da discordância em poucos grupos, e nesse caso o piso deve ser lido como faixa, e não como número.

Cabe registrar por que a chave não é o próprio vetor de entrada, que seria o caminho mais direto. A codificação reúne de propósito níveis de suporte insuficiente, e passa por isso a apresentar o mesmo vetor para ensaios que a planilha descreve como distintos. O piso estimado sobre o vetor de entrada incorpora essa diferença física como ruído, e cresce quanto mais a codificação simplifica, o que inverteria o sentido da medida. A chave é lida de arquivo, construída na preparação sobre as sete variáveis de configuração, os oito atributos candidatos anteriores ao filtro de suporte e o texto normalizado do motivo.

In [ ]:
# A chave de replicata e lida de arquivo, e nao derivada do vetor de entrada.
# Estimar o piso sobre o vetor de entrada superestima o ruido, porque a
# codificacao reune de proposito niveis de suporte insuficiente e passa a
# apresentar o mesmo vetor para ensaios fisicamente distintos.
CANDIDATOS_CHAVE = [
    "chave_replicata.json",
    os.path.join("..", "colabs_S4", "chave_replicata.json"),
    os.path.join("notebooks", "colabs_S4", "chave_replicata.json"),
]
_caminho = next((c for c in CANDIDATOS_CHAVE if os.path.exists(c)), None)
if _caminho is None:
    raise FileNotFoundError(
        "chave_replicata.json nao encontrado. Rode a Secao 9.1 do notebook de "
        "preparacao; o piso de ruido depende dela.")
CHAVE_REPLICATA = {int(k): v for k, v in
                   json.load(open(_caminho, encoding="utf-8"))["chave_por_run_idx"].items()}
print(f"Chave de replicata lida de {_caminho}: "
      f"{len(set(CHAVE_REPLICATA.values()))} configuracoes distintas")
print()


def piso_de_ruido(df):
    """Dispersao dos alvos entre ensaios de configuracao fisica identica.

    Retorna a media dos desvios-padrao por grupo, o estimador agrupado, o
    numero de grupos de repeticao e os graus de liberdade que os sustentam.
    """
    _, alvos = separar(df)
    # Mesma convencao de indices_da_dobra: run_idx quando presente no arquivo,
    # indice posicional caso contrario, pois o tratamento nao grava a coluna.
    run = (df["run_idx"] if "run_idx" in df.columns
           else pd.Series(np.arange(len(df)), index=df.index))
    chave = run.map(CHAVE_REPLICATA)
    assert chave.notna().all(), "ensaio sem chave de replicata"
    contagem = chave.value_counts()
    repetidas = contagem[contagem > 1].index
    if not len(repetidas):
        return np.nan, np.nan, 0, 0
    sub = df.assign(_chave=chave)[chave.isin(repetidas)]
    soma = graus = 0.0
    for _, grupo in sub.groupby("_chave"):
        for col in alvos:
            v = grupo[col].dropna()
            if len(v) > 1:
                soma += float(((v - v.mean()) ** 2).sum())
                graus += len(v) - 1
    return (float(sub.groupby("_chave")[alvos].std().mean().mean()),
            float(np.sqrt(soma / graus)) if graus else np.nan,
            int(len(repetidas)), int(graus))


linhas = []
for nome, df in sorted(datasets.items()):
    media_dp, agrupado, grupos, graus = piso_de_ruido(df)
    mae = df_ridge[df_ridge["dataset"] == nome]["MAE"].mean()
    linhas.append({"dataset": nome, "grupos": grupos, "gl": graus,
                   "media_dp": media_dp, "agrupado": agrupado,
                   "MAE": mae, "MAE/media_dp": mae / media_dp,
                   "MAE/agrupado": mae / agrupado})

piso = pd.DataFrame(linhas)
RUIDO_EXP = dict(zip(piso["dataset"], piso["agrupado"]))

print("Piso de ruido medido sobre a base atual, em graus Celsius:")
print(piso.round(2).to_string(index=False))
print()
print("gl = graus de liberdade somados sobre componentes e grupos de repeticao.")
print("A coluna agrupado e a adotada como piso; media_dp consta para comparacao")
print("com o procedimento anterior.")
print()
print("Valores anteriormente registrados, medidos sobre a chave de sete variaveis")
print("e sobre versao anterior da base, mantidos aqui apenas para contraste:")
ANTIGOS = {"oven_hmi": 1.64, "ct_knobs": 2.44, "main_vent_sys": 2.71,
           "ipc_cooling": 3.62, "mainback": 4.00, "ipc_comp": 4.61,
           "coil_temps": 5.50}
for nome in piso["dataset"]:
    novo = RUIDO_EXP[nome]
    print(f"  {nome:16s} antes {ANTIGOS[nome]:5.2f}   agora {novo:5.2f}   "
          f"variacao {100 * (novo - ANTIGOS[nome]) / ANTIGOS[nome]:+6.1f}%")

### Interpretação — Ridge baseline

O ruído experimental (`Ruido_exp`) serve como referência: ele representa a variabilidade inerente dos sensores, presente mesmo em medições repetidas perfeitas. Um MAE próximo ou abaixo desse valor indica que o modelo captura praticamente todo o sinal disponível nos dados. MAEs acima de duas vezes o ruído apontam dificuldade estrutural — seja pela ausência de variáveis explicativas relevantes, seja por comportamento não-linear que o modelo linear não consegue representar.

## 8. Lasso e ElasticNet

Lasso e ElasticNet são variações da regressão linear que adicionam penalizações sobre os coeficientes. O Lasso (regularização L1) leva coeficientes de features pouco relevantes a zero, funcionando como seleção automática de variáveis — útil quando apenas algumas das colunas binárias geradas pelo OHE realmente importam para um dado componente. O ElasticNet combina as regularizações L1 e L2, sendo mais estável quando há features correlacionadas, situação frequente após a codificação de categorias semelhantes.

O protocolo é idêntico ao do Ridge: mesma partição fixa, mesma função `indices_da_dobra`, mesma padronização, métricas acumuladas. O hiperparâmetro `alpha` — e, no caso do ElasticNet, o `l1_ratio` — é selecionado por cross-validation interno sobre o conjunto de treino de cada fold.

In [ ]:
from sklearn.linear_model import LassoCV

ALPHAS_LASSO = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

resultados_lasso = []
for nome, df in sorted(datasets.items()):
    X, cols_alvo = separar(df)
    for col in cols_alvo:
        y_col = df[col]
        if y_col.notna().sum() < MIN_PREDICOES: continue
        reais, preditos, alphas_usados = [], [], []
        for d in DOBRAS:
            idx_tr, idx_val = indices_da_dobra(df, d, y_col)
            if len(idx_tr) < MIN_TREINO or not idx_val: continue
            scaler = StandardScaler()
            X_tr_sc  = scaler.fit_transform(X.iloc[idx_tr])
            X_val_sc = scaler.transform(X.iloc[idx_val])
            cv_int = max(2, min(5, len(idx_tr) // 4))
            model = LassoCV(alphas=ALPHAS_LASSO, cv=cv_int, max_iter=10000)
            model.fit(X_tr_sc, y_col.iloc[idx_tr])
            reais.extend(y_col.iloc[idx_val].tolist())
            preditos.extend(model.predict(X_val_sc).tolist())
            alphas_usados.append(model.alpha_)
        if len(reais) < MIN_PREDICOES: continue
        reais = np.array(reais); preditos = np.array(preditos)
        resultados_lasso.append({
            "dataset": nome, "componente": col,
            **avaliar(reais, preditos, limite=limite_de(col)),
            "alpha_modal": float(max(set(alphas_usados), key=alphas_usados.count)),
        })

df_lasso = pd.DataFrame(resultados_lasso)
print(f"Componentes avaliados: {len(df_lasso)}")
print()
print(df_lasso.groupby("dataset")[["MAE","RMSE","R2"]].mean().round(3).to_string())

In [ ]:
from sklearn.linear_model import ElasticNetCV

L1_RATIOS   = [0.1, 0.3, 0.5, 0.7, 0.9]
ALPHAS_ENET = [0.001, 0.01, 0.1, 1.0, 10.0]

resultados_enet = []
for nome, df in sorted(datasets.items()):
    X, cols_alvo = separar(df)
    for col in cols_alvo:
        y_col = df[col]
        if y_col.notna().sum() < MIN_PREDICOES: continue
        reais, preditos, l1_usados = [], [], []
        for d in DOBRAS:
            idx_tr, idx_val = indices_da_dobra(df, d, y_col)
            if len(idx_tr) < MIN_TREINO or not idx_val: continue
            scaler = StandardScaler()
            X_tr_sc  = scaler.fit_transform(X.iloc[idx_tr])
            X_val_sc = scaler.transform(X.iloc[idx_val])
            cv_int = max(2, min(5, len(idx_tr) // 4))
            model = ElasticNetCV(l1_ratio=L1_RATIOS, alphas=ALPHAS_ENET, cv=cv_int, max_iter=10000)
            model.fit(X_tr_sc, y_col.iloc[idx_tr])
            reais.extend(y_col.iloc[idx_val].tolist())
            preditos.extend(model.predict(X_val_sc).tolist())
            l1_usados.append(model.l1_ratio_)
        if len(reais) < MIN_PREDICOES: continue
        reais = np.array(reais); preditos = np.array(preditos)
        resultados_enet.append({
            "dataset": nome, "componente": col,
            **avaliar(reais, preditos, limite=limite_de(col)),
            "l1_ratio_modal": float(max(set(l1_usados), key=l1_usados.count)),
        })

df_enet = pd.DataFrame(resultados_enet)
print(f"Componentes avaliados: {len(df_enet)}")
print()
print(df_enet.groupby("dataset")[["MAE","RMSE","R2"]].mean().round(3).to_string())

## 9. Random Forest

Random Forest é um ensemble de árvores de decisão que captura relações não-lineares e interações entre variáveis sem que essas relações precisem ser especificadas antecipadamente. Diferente dos modelos lineares, ele pode aprender que a combinação de dois fatores é relevante mesmo quando cada um isolado não é.

Os hiperparâmetros foram fixados para o contexto de datasets pequenos, com cerca de 40 amostras de treino por fold. `n_estimators=300` traz estabilidade com custo computacional baixo; `max_features="sqrt"` reduz a correlação entre as árvores ao sortear um subconjunto de features em cada divisão; `min_samples_leaf=2` evita overfitting ao exigir pelo menos dois pontos em cada folha; `random_state=42` garante reprodutibilidade. A padronização é mantida por uniformidade de protocolo, embora o Random Forest seja invariante a escala.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

RF_PARAMS = dict(n_estimators=300, max_features="sqrt", min_samples_leaf=2, random_state=42, n_jobs=-1)

resultados_rf = []
for nome, df in sorted(datasets.items()):
    X, cols_alvo = separar(df)
    for col in cols_alvo:
        y_col = df[col]
        if y_col.notna().sum() < MIN_PREDICOES: continue
        reais, preditos = [], []
        for d in DOBRAS:
            idx_tr, idx_val = indices_da_dobra(df, d, y_col)
            if len(idx_tr) < MIN_TREINO or not idx_val: continue
            scaler = StandardScaler()
            X_tr_sc  = scaler.fit_transform(X.iloc[idx_tr])
            X_val_sc = scaler.transform(X.iloc[idx_val])
            model = RandomForestRegressor(**RF_PARAMS)
            model.fit(X_tr_sc, y_col.iloc[idx_tr])
            reais.extend(y_col.iloc[idx_val].tolist())
            preditos.extend(model.predict(X_val_sc).tolist())
        if len(reais) < MIN_PREDICOES: continue
        reais = np.array(reais); preditos = np.array(preditos)
        resultados_rf.append({
            "dataset": nome, "componente": col,
            **avaliar(reais, preditos, limite=limite_de(col)),
        })

df_rf = pd.DataFrame(resultados_rf)
print(f"Componentes avaliados: {len(df_rf)}")
print()
print(df_rf.groupby("dataset")[["MAE","RMSE","R2"]].mean().round(3).to_string())

## 10. Comparação geral entre modelos

Os quatro modelos foram avaliados sobre a mesma partição fixa, o que permite comparação direta sem que diferenças nos resultados sejam atribuídas a variações no sorteio dos conjuntos de treino e validação.

In [ ]:
comparacao = pd.DataFrame({
    "Ridge"       : df_ridge.groupby("dataset")["MAE"].mean(),
    "Lasso"       : df_lasso.groupby("dataset")["MAE"].mean(),
    "ElasticNet"  : df_enet.groupby("dataset")["MAE"].mean(),
    "RandomForest": df_rf.groupby("dataset")["MAE"].mean(),
}).round(3)

comparacao["Ruido_exp"] = comparacao.index.map(RUIDO_EXP)
comparacao["Melhor"]    = comparacao[["Ridge","Lasso","ElasticNet","RandomForest"]].idxmin(axis=1)

print("MAE medio por dataset e modelo (menor e melhor):")
print()
print(comparacao.to_string())

### Consolidação por área

A tabela acima reúne a média simples do erro por área. A consolidação abaixo acrescenta a micro-média, que pondera cada componente pelo número de observações que o sustentam, e explicita a divergência entre as duas formas de agregar.

A distinção importa porque o IPC Comp reúne 37 das 74 colunas-alvo. A macro-média informa se o modelo funciona bem em todas as partes do produto, ao passo que a micro-média informa o erro típico de uma leitura qualquer e fica dominada pelas áreas com mais componentes medidos.

In [ ]:
for rotulo, tabela in [("Ridge", df_ridge), ("Lasso", df_lasso),
                       ("ElasticNet", df_enet), ("RandomForest", df_rf)]:
    resumo = consolidar(tabela)[["dataset", "componentes", "n_total",
                                 "MAE_macro", "MAE_micro", "R2_macro"]]
    print(f"--- {rotulo} ---")
    print(resumo.round(3).to_string(index=False))
    print()

### Interpretação final

Para ler a tabela, compare o MAE de cada modelo com o `Ruido_exp` da mesma linha. Um MAE abaixo do ruído indica que o modelo já é preciso o suficiente para o critério de segurança da Whirlpool. A coluna `Melhor` aponta qual modelo teve o menor MAE médio em cada área térmica — quando o Ridge já vence, modelos mais complexos não agregam valor para aquele dataset.

Na maior parte das áreas, os modelos lineares com regularização apresentam desempenho próximo ou superior ao Ridge simples, com ganhos que chegam a 11% em `ipc_comp`. O Random Forest se destaca em `mainback` e `oven_hmi`, superando os lineares em cerca de 10 a 13% — indício de relações não-lineares nessas regiões. A exceção é `coil_temps`, onde todos os modelos erram consistentemente acima de duas vezes o ruído, sugerindo que variáveis físicas relevantes para essa área não estão presentes nos dados coletados.

O próximo passo é aplicar os modelos selecionados ao conjunto de teste (ensaios em `IDX_TESTE`), verificar a aderência aos limites de segurança em `limites_ativos.csv` e documentar os resultados finais no relatório.

## 11. Decisão de aprovação contra o limite de segurança

As métricas anteriores medem distância em graus ou em pontos percentuais. Nenhuma delas informa se o modelo acerta a decisão que a engenharia efetivamente toma, que é aprovar ou reprovar um componente por comparação com o seu limite tabelado.

Dos dois tipos de discordância possíveis, apenas um é grave. Reprovar um componente que passaria é conservador e custa uma simulação adicional. Aprovar um componente que na realidade ultrapassa o limite é o erro que a ferramenta existe para evitar, e corresponde ao risco R06 da Seção 4.1.5.

A comparação abaixo ordena os modelos por essa métrica, e não pelo erro médio. Um modelo com erro em graus ligeiramente maior pode ser preferível caso deixe passar menos componentes reprovados, e essa possibilidade só se torna visível quando a decisão é medida diretamente.

In [ ]:
tabelas = [("Ridge", df_ridge), ("Lasso", df_lasso),
           ("ElasticNet", df_enet), ("RandomForest", df_rf)]

resumo_decisao = []
for rotulo, tabela in tabelas:
    com_limite = tabela[tabela.get("limite_C").notna()] if "limite_C" in tabela else tabela.iloc[0:0]
    if not len(com_limite):
        continue
    resumo_decisao.append({
        "modelo": rotulo,
        "componentes com limite": len(com_limite),
        "ensaios que reprovam": int(com_limite["n_reprova_real"].sum()),
        "falsos aprovados": int(com_limite["falso_aprovado"].sum()),
        "falsos reprovados": int(com_limite["falso_reprovado"].sum()),
        "acerto de decisao %": round(float(com_limite["acerto_decisao_%"].mean()), 1),
    })

resumo_decisao = pd.DataFrame(resumo_decisao).sort_values("falsos aprovados")
print("Decisao de aprovacao, ordenada pelo que importa (menos falsos aprovados e melhor):")
print()
print(resumo_decisao.to_string(index=False))
print()

melhor_erro = comparacao[["Ridge", "Lasso", "ElasticNet", "RandomForest"]].mean().idxmin()
melhor_decisao = resumo_decisao.iloc[0]["modelo"]
print(f"Menor erro medio em graus: {melhor_erro}")
print(f"Menos falsos aprovados:    {melhor_decisao}")
if melhor_erro != melhor_decisao:
    print()
    print("Os dois criterios apontam modelos diferentes. A escolha final precisa declarar")
    print("qual deles prevalece, decisao que pertence ao negocio e nao a estatistica.")

## 12. Busca ampla de algoritmos

A escolha dos quatro modelos anteriores partiu de conhecimento prévio sobre o problema. Para verificar se algum algoritmo fora dessa lista se sairia melhor, foi conduzida uma varredura com a biblioteca lazypredict, que ajusta cerca de quarenta regressores de uma vez.

A varredura foi aplicada sobre os componentes que efetivamente ultrapassam o limite de segurança, e não sobre todos, uma vez que é neles que o desempenho importa. A divisão utilizada é agrupada por configuração e retirada de dentro do conjunto de treino, de modo que o conjunto de teste permaneça reservado.

Duas ressalvas acompanham o resultado. A varredura utiliza uma única divisão, ao passo que o protocolo do projeto emprega dezessete dobras, de modo que a ordenação obtida é indicativa e não conclusiva. E a métrica da varredura é o erro em graus, que não é o critério de decisão adotado neste projeto. Por essas razões, os melhores colocados foram reavaliados sob o protocolo completo, e é essa reavaliação que sustenta a escolha.

A célula é opcional e falha de forma silenciosa caso a biblioteca não esteja instalada, de modo que o notebook permaneça executável sem ela.

In [ ]:
EXECUTAR_VARREDURA = False   # ligar para refazer a busca ampla

if EXECUTAR_VARREDURA:
    try:
        from lazypredict.Supervised import LazyRegressor
    except ImportError:
        print("lazypredict nao instalado. Para executar a varredura:")
        print("    !pip install lazypredict")
        LazyRegressor = None

    if LazyRegressor is not None:
        import warnings
        warnings.filterwarnings("ignore")

        DOBRAS_VAL = set(DOBRAS[-4:])   # divisao agrupada dentro do treino
        ranking = {}

        for nome, df in sorted(datasets.items()):
            X, cols_alvo = separar(df)
            criticos = [c for c in cols_alvo
                        if limite_de(c) is not None and (df[c] >= limite_de(c)).any()]
            for col in criticos[:4]:
                y_col = df[col]
                med = y_col.notna().to_numpy()
                tr = [i for i in range(len(df)) if med[i] and DOBRA.get(i) is not None
                      and DOBRA[i] not in DOBRAS_VAL]
                va = [i for i in range(len(df)) if med[i] and DOBRA.get(i) is not None
                      and DOBRA[i] in DOBRAS_VAL]
                if len(tr) < 10 or len(va) < 3:
                    continue
                reg = LazyRegressor(verbose=0, ignore_warnings=True, predictions=False)
                try:
                    modelos, _ = reg.fit(X.iloc[tr], X.iloc[va], y_col.iloc[tr], y_col.iloc[va])
                except Exception:
                    continue
                for alg, linha in modelos.iterrows():
                    ranking.setdefault(alg, []).append(float(linha.get("RMSE", np.nan)))

        resumo = pd.DataFrame([{"algoritmo": k, "avaliacoes": len(v),
                                "RMSE_medio": float(np.nanmean(v))}
                               for k, v in ranking.items()])
        print(resumo.sort_values("RMSE_medio").head(12).round(3).to_string(index=False))
else:
    print("Varredura desligada. Resultado obtido em 10 de setembro de 2026, sobre a")
    print("versao adotada, com 19 componentes criticos e dez primeiros colocados:")
    print()
    print("  AdaBoost 11,47 | GradientBoosting 11,88 | LassoLarsCV 11,95 | LarsCV 11,97")
    print("  ExtraTrees 12,18 | DecisionTree 12,19 | ExtraTree 12,21 | Bagging 12,52")
    print("  RandomForest 12,68 | LassoLarsIC 12,78")
    print()
    print("Tres observacoes decorrem da comparacao com a versao de 13 categorias:")
    print("  - o GradientBoosting sobe de quinto para segundo colocado, e a ordenacao da")
    print("    varredura passa a concordar com a decisao tomada sob o protocolo completo;")
    print("  - Lars e LarsCV deixam de divergir: na versao anterior alcancavam a ordem de")
    print("    10^15 por colinearidade, e aqui o LarsCV aparece em quarto lugar, o que")
    print("    evidencia que a reducao de dimensionalidade aliviou a instabilidade numerica;")
    print("  - o DummyRegressor, que prevê apenas a media, sai dos vinte primeiros, ao")
    print("    passo que na versao anterior superava metade da lista. A reducao retirou")
    print("    ruido, e nao sinal.")

## 13. Gradient Boosting

Os melhores colocados da varredura foram reavaliados sob o protocolo completo, com dezessete dobras e ordenados pelo número de falsos aprovados. O Gradient Boosting apresentou o melhor resultado nos dois critérios simultaneamente, com o menor erro em graus e o menor número de componentes reprovados que o modelo aprovaria.

Cabe registrar que a ordenação da varredura não se confirmou: o AdaBoost, primeiro colocado por erro em graus sobre uma única divisão, ficou em quarto na decisão sob o protocolo completo. É a razão pela qual a varredura foi tratada como indicativa e não como conclusiva.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

GB_PARAMS = dict(random_state=42)

resultados_gb = []
predicoes_gb = {}   # reaproveitado na calibracao da margem
for nome, df in sorted(datasets.items()):
    X, cols_alvo = separar(df)
    for col in cols_alvo:
        y_col = df[col]
        if y_col.notna().sum() < MIN_PREDICOES: continue
        reais, preditos, ensaios_val = [], [], []
        for d in DOBRAS:
            idx_tr, idx_val = indices_da_dobra(df, d, y_col)
            if len(idx_tr) < MIN_TREINO or not idx_val: continue
            scaler = StandardScaler()
            X_tr_sc  = scaler.fit_transform(X.iloc[idx_tr])
            X_val_sc = scaler.transform(X.iloc[idx_val])
            model = GradientBoostingRegressor(**GB_PARAMS)
            model.fit(X_tr_sc, y_col.iloc[idx_tr])
            reais.extend(y_col.iloc[idx_val].tolist())
            preditos.extend(model.predict(X_val_sc).tolist())
            # O ensaio de origem precisa acompanhar a predicao: a mascara de nulos
            # varia por componente, de modo que a posicao na lista nao identifica o
            # ensaio e nao serve para agrupar.
            ensaios_val.extend(df["run_idx"].iloc[idx_val].tolist()
                               if "run_idx" in df.columns else list(idx_val))
        if len(reais) < MIN_PREDICOES: continue
        reais = np.array(reais); preditos = np.array(preditos)
        limite = limite_de(col)
        resultados_gb.append({
            "dataset": nome, "componente": col,
            **avaliar(reais, preditos, limite=limite),
        })
        if limite is not None:
            predicoes_gb[(nome, col)] = (reais, preditos, limite,
                                         np.array(ensaios_val))

df_gb = pd.DataFrame(resultados_gb)
print(f"Componentes avaliados: {len(df_gb)}")
print()
print(df_gb.groupby("dataset")[["MAE","RMSE","R2"]].mean().round(3).to_string())
print()

com_limite = df_gb[df_gb["limite_C"].notna()]
print(f"Falsos aprovados: {int(com_limite['falso_aprovado'].sum())} "
      f"de {int(com_limite['n_reprova_real'].sum())} ensaios que reprovam")

## 14. Margem de segurança na decisão

A troca de algoritmo reduz os falsos aprovados, mas não os elimina, e a razão é estrutural. O modelo é ajustado para acertar a média condicional, e nos ensaios que ultrapassam o limite a média condicional está abaixo do valor observado. O diagnóstico é direto: o viés médio geral é de mais 1,7 grau, ao passo que nos ensaios que ultrapassam o limite ele é de menos 17,1 graus, contra uma margem típica de estouro de 7,7 graus. O modelo subestima justamente onde a subestimação é perigosa, e nenhum algoritmo treinado por erro simétrico corrige isso.

O que corrige é deslocar o limiar da decisão. Em lugar de comparar a predição com o limite, compara-se a predição acrescida de uma margem de segurança. A operação não altera o modelo, apenas a regra de decisão construída sobre ele, e converte falsos aprovados em falsos reprovados.

A conversão é desejável porque os dois erros têm custos distintos. Reprovar indevidamente custa uma simulação de verificação. Aprovar indevidamente deixa passar um superaquecimento, que é o risco R06 da Seção 4.1.5 e a razão de existir da ferramenta.

A célula abaixo calibra essa margem, permitindo que a escolha seja feita com o custo de cada opção à vista.

In [ ]:
def decisao_com_margem(tabela, predicoes_por_componente, margem):
    """Recontabiliza a decisao deslocando o limiar pela margem indicada."""
    fa = fr = acima = 0
    for (nome, col), (reais, preditos, limite, _) in predicoes_por_componente.items():
        real = reais >= limite
        prevista = (preditos + margem) >= limite
        fa += int(np.sum(real & ~prevista))
        fr += int(np.sum(~real & prevista))
        acima += int(real.sum())
    return fa, fr, acima


# As predicoes ja foram guardadas na Secao 13: reajustar o modelo aqui
# duplicaria o trabalho sem alterar resultado algum.
print(f"Componentes com limite disponiveis: {len(predicoes_gb)}")

linhas = []
for margem in [0, 3, 5, 8, 10, 12, 15, 20]:
    fa, fr, acima = decisao_com_margem(df_gb, predicoes_gb, float(margem))
    linhas.append({
        "margem (C)": margem,
        "falsos aprovados": fa,
        "falsos reprovados": fr,
        "estouros detectados %": round(100 * (acima - fa) / acima, 1),
    })

calibracao = pd.DataFrame(linhas)
print("Calibracao da margem de seguranca, Gradient Boosting:")
print()
print(calibracao.to_string(index=False))
print()
_, _, total_acima = decisao_com_margem(df_gb, predicoes_gb, 0.0)
total_predicoes = sum(len(v[0]) for v in predicoes_gb.values())
print(f"Ensaios que de fato ultrapassam o limite: {total_acima} de {total_predicoes} predicoes")

### Leitura da calibração, na unidade em que a decisão é tomada

A tabela acima conta falsos aprovados por componente. Essa não é a unidade em que a engenharia decide, e a diferença altera a conclusão.

Um fogão reprova se qualquer um dos seus componentes ultrapassa o limite. Medida por ensaio, a detecção é de 94,9 por cento já sem margem alguma, e não de 57,5 por cento como a contagem por componente sugere. A razão é que os falsos aprovados se concentram em designs que já são reprovados por outro componente, de modo que não geram erro de decisão. Com margem de 8 graus, a detecção por ensaio alcança 100 por cento dos 39 designs que reprovam.

O custo, nessa unidade, é nulo. Os falsos reprovados por ensaio permanecem em sete, invariáveis de margem zero a vinte graus, pois as reprovações indevidas de componente também se acumulam em designs que já seriam reprovados. Os 285 falsos reprovados por componente da margem de dez graus não produzem uma única verificação adicional de fogão.

A margem, portanto, é adotada: ela eleva a detecção sem custo mensurável.

### A pergunta binária não é a entrega útil

A mesma medição revela um limite da formulação. Dos 46 ensaios avaliados, 39 reprovam de fato e 7 passam, e o modelo sinaliza todos os 7 como reprovados. Como classificador de aprovação do fogão inteiro, ele não discrimina: responde reprovado a quase tudo, e acerta 85 por cento por essa razão, e não por mérito.

A causa é estrutural. Com 68 componentes por ensaio, basta um estourar para o design reprovar, e a maior parte dos designs da base possui ao menos um estouro. A pergunta se este fogão passa tem resposta quase sempre negativa, e um veredito agregado informa pouco.

O valor da ferramenta está na lista, e não no veredito. As métricas abaixo medem essa lista.

In [ ]:
# Qualidade da lista de componentes sinalizados, por ensaio.
# Reaproveita as predicoes guardadas na Secao 13.

MARGEM_DECISAO = 8.0

registros = []
for (area, col), (reais, preditos, limite, ensaios) in predicoes_gb.items():
    for real, previsto, ensaio in zip(reais, preditos, ensaios):
        registros.append({"area": area, "componente": col, "ensaio": int(ensaio),
                          "real": real, "previsto": previsto, "limite": limite})
registros = pd.DataFrame(registros)
registros["estoura_real"] = registros["real"] >= registros["limite"]
registros["sinalizado"] = (registros["previsto"] + MARGEM_DECISAO) >= registros["limite"]
registros["folga_prevista"] = registros["limite"] - registros["previsto"]
registros["folga_real"] = registros["limite"] - registros["real"]

por_ensaio = []
for ensaio, g in registros.groupby("ensaio"):
    reais = set(g.loc[g["estoura_real"], "componente"])
    sinal = set(g.loc[g["sinalizado"], "componente"])
    if not reais:
        continue
    por_ensaio.append({
        "componentes": len(g),
        "estouros_reais": len(reais),
        "sinalizados": len(sinal),
        "recall": len(reais & sinal) / len(reais),
        "precisao": len(reais & sinal) / len(sinal) if sinal else np.nan,
    })
por_ensaio = pd.DataFrame(por_ensaio)

print(f"Ensaios com ao menos um estouro real: {len(por_ensaio)}")
print(f"Componentes por ensaio:     mediana {int(por_ensaio.componentes.median())}")
print(f"Estouros reais por ensaio:  mediana {int(por_ensaio.estouros_reais.median())}")
print(f"Sinalizados por ensaio:     mediana {int(por_ensaio.sinalizados.median())}")
print()
print(f"Recall medio   (dos estouros reais, quantos foram sinalizados): "
      f"{por_ensaio.recall.mean():.1%}")
print(f"Precisao media (dos sinalizados, quantos estouram de fato):     "
      f"{por_ensaio.precisao.mean():.1%}")
print()
print(f"A engenharia verifica {por_ensaio.sinalizados.median():.0f} componentes de "
      f"{por_ensaio.componentes.median():.0f}, ou seja "
      f"{100*por_ensaio.sinalizados.median()/por_ensaio.componentes.median():.0f} por cento,")
print("e nessa lista encontra cerca de tres quartos dos superaquecimentos reais.")
print()

# Ordenacao por risco, sem regra de corte
posicoes = []
for ensaio, g in registros.groupby("ensaio"):
    reais = g[g["estoura_real"]]
    if not len(reais):
        continue
    pior = reais.loc[reais["folga_real"].idxmin(), "componente"]
    ordem = g.sort_values("folga_prevista")["componente"].tolist()
    posicoes.append(ordem.index(pior) + 1)
posicoes = np.array(posicoes)

print("Ordenacao por folga prevista, sem limiar algum:")
for k in [1, 3, 5, 10]:
    print(f"  componente mais critico entre os {k:2d} primeiros: "
          f"{100*np.mean(posicoes <= k):5.1f}% dos ensaios")
print(f"  posicao mediana do componente mais critico: {int(np.median(posicoes))}")

### O que estas métricas sustentam

Três conclusões decorrem da medição na unidade correta.

A ferramenta reduz o trabalho de verificação em cerca de noventa e quatro por cento, ao apontar quatro componentes entre sessenta e oito, e nessa lista curta encontra aproximadamente três quartos dos superaquecimentos reais. A precisão em torno de metade significa que parte da lista é alarme falso, o que é aceitável quando a lista tem quatro itens e o erro oposto deixa passar uma falha térmica.

A ordenação por folga prevista sustenta a ferramenta mesmo sem regra de corte. O componente que de fato está mais próximo do limite ocupa a segunda posição da lista, em mediana, e aparece entre os cinco primeiros em setenta e dois por cento dos ensaios. Essa propriedade não depende da calibração da margem e é, por isso, a mais robusta das evidências reunidas.

O veredito binário sobre o fogão inteiro não deve compor a entrega. Ele é quase degenerado nesta base, e substituí-lo pela lista ordenada de componentes em risco descreve melhor o que a ferramenta faz e o que a engenharia utiliza.

Registra-se a limitação amostral: a conclusão sobre os designs aprovados apoia-se em sete casos, número insuficiente para estimar a taxa de reprovação indevida com precisão útil.

## 15. Alternativas avaliadas e descartadas

Duas arquiteturas alternativas foram testadas sob o mesmo protocolo e descartadas. As medições desta seção foram realizadas sobre a versão de 13 categorias, anterior à adoção da versão com perfil de perdas. A comparação entre as duas versões, registrada na Seção 4.3.1 da documentação, resultou em diferença vinte vezes inferior ao desvio da própria estimativa, de modo que as conclusões aqui registradas, que dizem respeito a arquitetura e não a conjunto de features, permanecem válidas. O registro é feito porque um resultado negativo medido tem valor de evidência: ele fecha um caminho que de outro modo seria retomado, e sustenta por que a arquitetura adotada é a que está adotada.

### Modelo compartilhado por área

A hipótese era de que os componentes de uma mesma área respondem de forma parecida à configuração, de modo que empilhá-los num único modelo permitiria estimar o efeito do design com muito mais dados. No IPC Comp, o conjunto passaria de 46 linhas para cerca de 1.700.

Foram testadas duas implementações. Na primeira, os componentes foram empilhados com um indicador próprio ao lado das colunas de configuração. Na segunda, o alvo passou a ser o desvio em relação à média do componente, calculada apenas sobre o treino de cada dobra, de modo que a variação entre componentes saísse do alvo e a capacidade do modelo se concentrasse no efeito da configuração.

Ambas falharam, e a segunda falhou mais que a primeira. A detecção de estouros caiu de 57,5 por cento, na arquitetura por componente, para 8,3 e 2,1 por cento respectivamente. As predições passaram a colar na média de cada componente.

A conclusão é substantiva e não apenas técnica: os componentes de uma mesma área não compartilham uma resposta comum à configuração. Um transistor e um capacitor da mesma placa reagem de formas diferentes à abertura do deflector, e impor uma função de resposta única anula as duas. O empilhamento só ajudaria caso a resposta fosse compartilhada, e a medição mostra que não é. A arquitetura de um modelo por componente fica, assim, justificada por evidência, e não apenas por conveniência.

### Regressão quantílica

A hipótese era de que prever um quantil superior, em lugar da média condicional, seria uma forma mais principiada de obter folga de segurança que a margem fixa da Seção 14, por adaptar-se à incerteza de cada componente em vez de aplicar o mesmo deslocamento a todos.

Foram testados os quantis 0,75, 0,85, 0,90 e 0,95, e comparados com a margem fixa nos mesmos níveis de falsos aprovados. As duas abordagens percorrem a mesma curva de compromisso: o quantil 0,95 produz 32 falsos aprovados e 287 falsos reprovados, ao passo que a margem de 10 graus produz 33 e 285. O quantil 0,90 produz 47 e 201, contra 43 e 225 da margem de 8 graus.

Não havendo vantagem mensurável, a margem fixa foi mantida por ser mais simples de explicar à parceira, de auditar e de calibrar. Registra-se que a alternativa foi testada, e não apenas descartada por preferência.

In [ ]:
# Resultados obtidos na execucao de 10 de setembro de 2026, sob o protocolo completo
# de dezessete dobras. A decisao e avaliada sem margem, exceto onde indicado.

alternativas = pd.DataFrame([
    {"abordagem": "Por componente, sem margem",   "MAE": 7.54,  "falsos_aprovados": 82,
     "falsos_reprovados": 60,  "deteccao_%": 57.5},
    {"abordagem": "Por componente, margem 8 C",   "MAE": 7.54,  "falsos_aprovados": 43,
     "falsos_reprovados": 225, "deteccao_%": 77.7},
    {"abordagem": "Por componente, margem 10 C",  "MAE": 7.54,  "falsos_aprovados": 33,
     "falsos_reprovados": 285, "deteccao_%": 82.9},
    {"abordagem": "Quantilico 0,90",              "MAE": 11.89, "falsos_aprovados": 47,
     "falsos_reprovados": 201, "deteccao_%": 75.6},
    {"abordagem": "Quantilico 0,95",              "MAE": 13.66, "falsos_aprovados": 32,
     "falsos_reprovados": 287, "deteccao_%": 83.4},
    {"abordagem": "Compartilhado com indicador",  "MAE": 9.30,  "falsos_aprovados": 177,
     "falsos_reprovados": None, "deteccao_%": 8.3},
    {"abordagem": "Compartilhado centrado",       "MAE": 9.30,  "falsos_aprovados": 189,
     "falsos_reprovados": 28,  "deteccao_%": 2.1},
])

print("Alternativas avaliadas, ordenadas por deteccao de estouros:")
print()
print(alternativas.sort_values("deteccao_%", ascending=False).to_string(index=False))
print()
print("Leitura: o quantilico empata com a margem fixa nos mesmos niveis de falsos")
print("aprovados, e o compartilhado colapsa a deteccao em ambas as implementacoes.")

### Ajustes de configuração avaliados

Além das duas arquiteturas alternativas, seis configurações do modelo adotado foram comparadas sob o mesmo protocolo, com a decisão avaliada sob margem de 10 graus. As contagens de falsos aprovados e reprovados desta seção não são comparáveis às das Seções 14, 21 e 23, que usam a margem de 8 graus adotada no projeto. Nenhuma delas melhorou o resultado de forma mensurável.

Três conclusões decorrem da comparação.

A regularização do Gradient Boosting piora o desempenho. Reduzir a profundidade para dois, elevar o número de árvores para trezentas, baixar a taxa de aprendizado para 0,05 e introduzir subamostragem degrada o erro absoluto médio em quatorze por cento. Os parâmetros padrão já são adequados ao tamanho da amostra, e a intuição de que uma base pequena exige mais regularização não se confirmou.

A remoção das colunas de perdas também piora. O erro sobe de 7,54 para 7,97 graus e os falsos aprovados de 33 para 39. As treze colunas carregam, portanto, informação que não está contida no tipo de ensaio, ao menos para modelos baseados em árvores. O achado é relevante para a comparação pendente entre as duas versões da base, e aponta em sentido contrário à expectativa de que reduzir a dimensionalidade traria ganho.

O ensemble das três melhores famílias produz diferença de 0,043 grau no erro absoluto médio, valor quase dez vezes inferior ao desvio de 0,35 grau da própria estimativa sob validação cruzada. A diferença está abaixo da precisão da medição e não justifica triplicar o custo de treino.

Registra-se ainda uma verificação de premissa. O limiar mínimo de linhas para ajustar um modelo em uma dobra, fixado em oito, nunca é atingido: o menor conjunto de treino observado tem vinte e três linhas e a mediana é de quarenta e duas. Apenas oito dos 1.239 ajustes possuem menos linhas que colunas de entrada, o que explica por que apenas um estimador da varredura se recusou a rodar. Elevar o limiar para vinte produz resultado idêntico, confirmando que ele não é restritivo.

In [ ]:
# Resultados obtidos na execucao de 10 de setembro de 2026, sob o protocolo completo.
# A decisao e avaliada com margem de 10 C em todas as configuracoes.

configuracoes = pd.DataFrame([
    {"configuracao": "GB padrao (adotada)",          "MAE": 7.543,
     "falsos_aprovados": 33, "falsos_reprovados": 285, "deteccao_%": 82.9},
    {"configuracao": "GB regularizado",              "MAE": 8.572,
     "falsos_aprovados": 42, "falsos_reprovados": 290, "deteccao_%": 78.2},
    {"configuracao": "GB sem colunas de perdas",     "MAE": 7.966,
     "falsos_aprovados": 39, "falsos_reprovados": 262, "deteccao_%": 79.8},
    {"configuracao": "GB regularizado sem perdas",   "MAE": 8.744,
     "falsos_aprovados": 45, "falsos_reprovados": 244, "deteccao_%": 76.7},
    {"configuracao": "Ensemble GB, ExtraTrees, Bag", "MAE": 7.500,
     "falsos_aprovados": 32, "falsos_reprovados": 288, "deteccao_%": 83.4},
    {"configuracao": "Minimo de treino elevado",     "MAE": 7.543,
     "falsos_aprovados": 33, "falsos_reprovados": 285, "deteccao_%": 82.9},
])

print("Configuracoes avaliadas, com margem de 10 C:")
print()
print(configuracoes.to_string(index=False))
print()

adotada = configuracoes.iloc[0]["MAE"]
melhor = configuracoes["MAE"].min()
print(f"Melhor MAE observado: {melhor:.3f} C, contra {adotada:.3f} C da configuracao adotada.")
print(f"Diferenca: {adotada - melhor:.3f} C, frente a um desvio de estimativa de 0,35 C")
print("sob validacao cruzada. A diferenca esta abaixo da precisao da medicao.")

## 16. Valores extremos: investigação e decisão

A sinalização de valores extremos permaneceu pendente desde a revisão do pré-processamento. A verificação foi conduzida aqui, e a conclusão é que eles devem ser mantidos.

São 92 leituras além de três vezes o intervalo interquartil do próprio componente, o que corresponde a 2,3 por cento da base. A questão a resolver era se constituem erro de medição, caso em que distorceriam o ajuste, ou fenômeno físico legítimo, caso em que removê-los eliminaria justamente o sinal que a ferramenta precisa aprender.

### Os extremos correspondem ao modo pyro com variação de ventilação

Os 92 casos concentram-se em 17 dos 56 ensaios, e os oito ensaios com maior número deles são todos do modo pyro, reunindo 78 das 92 ocorrências. O motivo registrado na planilha original descreve, em quase todos, alteração física deliberada: Aluminium Plate Pyro Blower On Medium, Blower On High, Fan velocity 1500 RPM, Double Flow on Blower.

A distribuição corrobora a leitura. Dos 92 casos, 64 estão acima e 28 abaixo do corpo da distribuição, com desvio mediano de mais 16,3 graus em relação à mediana do componente. A assimetria é coerente com um fenômeno em que a ventilação aquece certas regiões e resfria outras, e não com erro espalhado em torno do valor central. A concentração por área reforça: Mainback, Coil Temps e Main Vent Sys reúnem 68 dos 92 casos, que são as regiões de exaustão e das bobinas, diretamente afetadas pelo soprador.

### Os extremos não são os estouros de limite

Esta foi a verificação decisiva. Dos 256 estouros de limite presentes na base, apenas 16, ou 6,2 por cento, são classificados como valores extremos.

Os superaquecimentos que a ferramenta precisa detectar não estão na cauda da distribuição de cada componente. São leituras ordinárias para aquele sensor, que ultrapassam um limite baixo. Um transistor com limite de 110 graus estoura em temperaturas perfeitamente comuns para ele.

Duas consequências decorrem daí. A remoção dos extremos não destruiria o sinal de estouro, ao contrário do que se poderia temer; mas também não corrigiria o viés de subestimação nos ensaios que reprovam, uma vez que esse viés se manifesta nos 94 por cento de estouros que não são extremos.

### Decisão e pendência aberta

Os valores extremos são mantidos. Removê-los retiraria do treino os ensaios que representam o modo de operação mais severo do produto, e o modelo perderia desempenho exatamente onde ele é mais necessário. A pendência de tratamento de outliers fica encerrada por esta verificação, e não por aplicação de um critério automático.

Registra-se, contudo, uma constatação de maior alcance. Os motivos dos ensaios citam de forma explícita a intensidade do soprador, em níveis como médio, alto e 1500 rotações por minuto. Essa informação não integra o conjunto de features: existe a direção do soprador, mas não a sua intensidade. Caso a variação de vazão seja o que produz esses 78 casos extremos, o modelo está sendo cobrado por prever um efeito cuja causa não lhe é fornecida.

A extração dessa variável para uma coluna própria constitui o pedido de maior valor a ser encaminhado à parceira, por ser a primeira proposta de feature nova com fundamento físico desde o início da modelagem.

In [ ]:
# Verificacao dos valores extremos, por componente, alem de tres vezes o IQR.
# O motivo de cada ensaio vem da planilha original, cuja coluna foi descartada na
# preparacao por nao descrever caracteristica fisica; aqui ela serve de evidencia.

MULT_EXTREMO_DIAG = 3.0

extremos = []
for nome, df in sorted(datasets.items()):
    X, cols_alvo = separar(df)
    rc = [c for c in df.columns if c.startswith("run_case_")]
    tipo = df[rc].idxmax(axis=1).str.replace("run_case_", "", regex=False)
    for col in cols_alvo:
        s = df[col].dropna()
        if len(s) < 10:
            continue
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        alto, baixo = q3 + MULT_EXTREMO_DIAG * iqr, q1 - MULT_EXTREMO_DIAG * iqr
        L = limite_de(col)
        for i in df.index:
            v = df[col].iloc[i]
            if pd.isna(v) or baixo <= v <= alto:
                continue
            extremos.append({"area": nome, "componente": col, "ensaio": int(i),
                             "valor": float(v), "lado": "alto" if v > alto else "baixo",
                             "desvio_da_mediana": float(v - s.median()),
                             "run_case": tipo.iloc[i],
                             "acima_do_limite": bool(L is not None and v >= L),
                             "tem_limite": L is not None})

extremos = pd.DataFrame(extremos)
print(f"Valores extremos: {len(extremos)} de "
      f"{sum(df[separar(df)[1]].notna().sum().sum() for df in datasets.values())} leituras")
print()
print("Por lado:", extremos["lado"].value_counts().to_dict())
print(f"Desvio mediano em relacao a mediana do componente: "
      f"{extremos['desvio_da_mediana'].median():+.2f} C")
print()
print("Por tipo de ensaio:")
print(extremos["run_case"].value_counts().to_string())
print()
print("Por area:")
print(extremos["area"].value_counts().to_string())
print()

# A verificacao decisiva
total_estouros = 0
for nome, df in datasets.items():
    for col in separar(df)[1]:
        L = limite_de(col)
        if L is not None:
            total_estouros += int((df[col] >= L).sum())

estouros_extremos = int(extremos["acima_do_limite"].sum())
print("=" * 70)
print(f"Estouros de limite na base:          {total_estouros}")
print(f"Desses, classificados como extremos: {estouros_extremos} "
      f"({100 * estouros_extremos / total_estouros:.1f}%)")
print("=" * 70)
print()
print("A fracao baixa indica que os estouros nao estao na cauda da distribuicao de")
print("cada componente: sao leituras ordinarias que ultrapassam um limite baixo.")
print("Remover os extremos nao destruiria esse sinal, nem corrigiria o vies de")
print("subestimacao, que se manifesta nos estouros que nao sao extremos.")

## 17. Comparação entre as duas versões da base

Duas versões dos conjuntos foram preparadas, diferindo exclusivamente no tratamento da coluna de perdas. Na primeira ela conserva as 13 categorias originais, resultando em 27 colunas de entrada; na segunda é reduzida ao perfil de perdas, resultando em 17.

A comparação é possível sem novo sorteio porque a partição foi construída sobre a chave do perfil, que constitui um agrupamento mais grosso da chave de 13 categorias. Uma partição construída sobre a chave detalhada valeria apenas para uma das versões, conforme verificado no notebook de ausência de vazamento.

### Por que a comparação é pareada

Cada componente é avaliado nas duas versões sobre as mesmas dobras e os mesmos ensaios. A comparação de médias agregadas desperdiçaria essa correspondência e teria menor poder de detectar diferença, uma vez que a variação entre componentes, que é grande, se somaria à incerteza. O teste pareado elimina essa fonte de variação ao comparar cada componente consigo mesmo.

### Resultado

Sobre os 74 componentes, o erro absoluto médio foi de 7,543 graus na versão de 13 categorias e de 7,560 na versão com perfil. A versão com perfil apresentou desempenho melhor em 36 componentes e pior em 38, com diferença mediana de 0,004 grau.

O teste t pareado resultou em p igual a 0,848 e o teste de Wilcoxon em p igual a 0,777. O intervalo de confiança de 95 por cento para a diferença média é de menos 0,162 a mais 0,197 grau, e contém zero com folga em ambos os lados.

As duas versões são, portanto, estatisticamente indistinguíveis. Nenhuma afirmação de ganho de desempenho por qualquer uma delas se sustenta.

### Decisão

Adotou-se a versão com perfil de perdas, por três razões, nenhuma delas de desempenho.

A primeira é a parcimônia, critério registrado na Seção 4.3.1 da documentação antes de qualquer resultado ser conhecido: em caso de empate, prefere-se a versão de menor dimensionalidade, por reduzir o risco de sobreajuste em uma base com 46 linhas de treino.

A segunda é a estabilidade numérica. Na varredura ampla de algoritmos, os estimadores Lars e LarsCV divergiram para a ordem de dez elevado a quinze sobre a versão de 13 categorias, efeito da colinearidade das colunas indicadoras, e convergiram normalmente sobre a versão com perfil.

A terceira é a margem para novas variáveis. A extração de atributos físicos a partir do motivo registrado de cada ensaio, discutida como proposta de trabalho futuro, acrescenta colunas ao conjunto de entrada, e a versão de menor dimensionalidade acomoda esse acréscimo sem agravar a razão entre variáveis e observações.

### Nota posterior

A versão adotada pelo projeto passou a ser a de treze categorias. A comparação registrada nesta seção permanece válida como medida, e continua indicando empate entre as duas versões; o que mudou foi o critério de desempate, conforme se explica na célula de configuração.

In [ ]:
EXECUTAR_PAREADO = False   # ligar para refazer a comparacao entre as versoes

if EXECUTAR_PAREADO:
    from scipy import stats

    def _mae_por_componente(sufixo):
        saida = {}
        origem = PASTA_DADOS if sufixo == "_13cat" else PASTA_DADOS_PERFIL
        pasta = os.path.dirname(resolve(origem, f"dataset_mainback{sufixo}.csv"))
        for nome in sorted(datasets):
            df = pd.read_csv(os.path.join(pasta, f"dataset_{nome}{sufixo}.csv"))
            X, cols_alvo = separar(df)
            for col in cols_alvo:
                y_col = df[col]
                if y_col.notna().sum() < MIN_PREDICOES:
                    continue
                reais, preditos = [], []
                for d in DOBRAS:
                    idx_tr, idx_val = indices_da_dobra(df, d, y_col)
                    if len(idx_tr) < MIN_TREINO or not idx_val:
                        continue
                    escala = StandardScaler()
                    modelo = GradientBoostingRegressor(**GB_PARAMS)
                    modelo.fit(escala.fit_transform(X.iloc[idx_tr]), y_col.iloc[idx_tr])
                    preditos.extend(modelo.predict(escala.transform(X.iloc[idx_val])))
                    reais.extend(y_col.iloc[idx_val])
                if len(reais) >= MIN_PREDICOES:
                    saida[(nome, col)] = float(np.mean(np.abs(np.array(preditos)
                                                             - np.array(reais))))
        return saida

    a = _mae_por_componente("_13cat")
    b = _mae_por_componente("_perfil")
    comuns = sorted(set(a) & set(b))
    va = np.array([a[k] for k in comuns])
    vb = np.array([b[k] for k in comuns])
    dif = vb - va

    print(f"componentes comparados: {len(comuns)}")
    print(f"MAE macro  13cat {va.mean():.3f} | perfil {vb.mean():.3f} | "
          f"diferenca {dif.mean():+.3f} C")
    print(f"perfil melhor em {int((dif < 0).sum())}, pior em {int((dif > 0).sum())}")
    print(f"teste t pareado: p={stats.ttest_rel(vb, va).pvalue:.3f} | "
          f"Wilcoxon: p={stats.wilcoxon(vb, va).pvalue:.3f}")
    ic = stats.t.interval(0.95, len(dif) - 1, loc=dif.mean(), scale=stats.sem(dif))
    print(f"IC 95% da diferenca media: [{ic[0]:+.3f}, {ic[1]:+.3f}] C")
else:
    print("Comparacao desligada. Resultado obtido em 10 de setembro de 2026,")
    print("sobre os 74 componentes, com o mesmo modelo e a mesma particao:")
    print()
    print("  MAE macro       13cat 7,543  |  perfil 7,560  |  diferenca +0,017 C")
    print("  perfil melhor em 36 componentes, pior em 38")
    print("  diferenca mediana por componente: +0,004 C")
    print()
    print("  teste t pareado : p = 0,848")
    print("  Wilcoxon        : p = 0,777")
    print("  IC 95% da diferenca media: [-0,162 , +0,197] C")
    print()
    print("O intervalo contem zero com folga em ambos os lados, e os dois valores de p")
    print("ficam acima de 0,77. As versoes sao estatisticamente indistinguiveis, e a")
    print("escolha entre elas nao pode ser justificada por desempenho.")

## 18. As placas de alumínio, reportadas em separado

As quatro placas de alumínio são retiradas da média da área e reportadas à parte. A separação não busca melhorar o número apresentado, e sim impedir que dois problemas de natureza distinta sejam lidos como um só.

Três razões a sustentam.

A primeira é que as placas não constituem componentes da área em que se encontram. A planilha original não declara área para nenhuma das quatro, e elas herdaram Coil Temps por preenchimento a partir da linha anterior, conforme registrado na preparação. O limite delas é de 150 graus, contra os 220 dos componentes Coil, o que indica objeto térmico diferente.

A segunda é que o erro delas não mede predição, e sim extrapolação. Os seis ensaios acima de 175 graus concentram-se em duas dobras, e os quatro mais extremos reúnem-se em uma só, a que contém os ensaios de placa de plástico em broil c2. Quando essa dobra é a validação, restam no treino apenas ensaios de plástico em broil qualitative e broil c1, todos em torno de 122 graus. O modelo é cobrado por prever 220 graus sem dispor de um único exemplo do regime, e o erro médio nas predições cujo valor real supera o maior valor visto no treino é de 57 graus, contra 3,4 graus nas demais.

A terceira é que a informação necessária está presente na base, o que descarta a hipótese de variável faltante. A coluna de perdas explica de 75 a 86 por cento da variância das placas, o caso de ensaio de 71 a 81 por cento, e o material do suporte de bobina de 19 a 40 por cento. Na faixa de 100 a 150 graus, onde há densidade de observações, o erro absoluto médio é de 6,9 graus.

Reunir as placas à média da área produz duas leituras equivocadas ao mesmo tempo: atribui aos demais componentes de Coil Temps uma dificuldade que não é deles, e dilui em uma média a única falha que compromete a decisão de segurança.

In [ ]:
# As metricas de erro sao recompostas a partir das medias por componente,
# ponderadas pelo numero de predicoes, o que reproduz o valor micro exato.
# As metricas de decisao vem de registros, que carrega a margem de seguranca.
E_PLACA = "Aluminium Plate"


def _erro(d):
    n = d["n"].sum()
    if not n:
        return {}
    return {"componentes": len(d), "predicoes": int(n),
            "MAE": float((d["MAE"] * d["n"]).sum() / n),
            "MAPE_%": float((d["MAPE_%"] * d["n"]).sum() / n),
            "dentro_10_%": float((d["dentro_10_%"] * d["n"]).sum() / n)}


def _decisao(d):
    estoura = d["estoura_real"]
    return {"estouros": int(estoura.sum()),
            "falso_aprovado": int((estoura & ~d["sinalizado"]).sum()),
            "falso_reprovado": int((~estoura & d["sinalizado"]).sum())}


def _bloco(rotulo, mask_gb, mask_reg):
    return {"recorte": rotulo, **_erro(df_gb[mask_gb]),
            **_decisao(registros[mask_reg])}


placa_gb = df_gb["componente"].str.contains(E_PLACA)
placa_reg = registros["componente"].str.contains(E_PLACA)
coil_gb = df_gb["dataset"] == "coil_temps"
coil_reg = registros["area"] == "coil_temps"

recortes = pd.DataFrame([
    _bloco("todos os componentes", df_gb["componente"].notna(),
           registros["componente"].notna()),
    _bloco("sem as placas de aluminio", ~placa_gb, ~placa_reg),
    _bloco("somente as placas", placa_gb, placa_reg),
    _bloco("coil_temps completo", coil_gb, coil_reg),
    _bloco("coil_temps sem as placas", coil_gb & ~placa_gb, coil_reg & ~placa_reg),
])
print(f"Metricas por recorte, margem de seguranca de {MARGEM_DECISAO:.0f} C:")
print(recortes.round(2).to_string(index=False))
print()

_p, _t = recortes.iloc[2], recortes.iloc[0]
print(f"As placas somam {_p['predicoes']} das {_t['predicoes']} predicoes, ou "
      f"{100 * _p['predicoes'] / _t['predicoes']:.1f} por cento do total, e "
      f"respondem por {_p['falso_aprovado']} dos {_t['falso_aprovado']} falsos "
      f"aprovados, ou {100 * _p['falso_aprovado'] / _t['falso_aprovado']:.0f} por cento.")
print()

# O erro das placas por faixa de temperatura separa a regiao com densidade de
# observacoes das faixas que exigem extrapolacao.
_pl = registros[placa_reg].copy()
_pl["erro"] = _pl["previsto"] - _pl["real"]
_pl["faixa"] = pd.cut(_pl["real"], [0, 100, 150, 200, 250])
print("Erro das placas por faixa do valor real:")
print(_pl.groupby("faixa", observed=True).agg(
    n=("erro", "size"), vies=("erro", "mean"),
    MAE=("erro", lambda s: s.abs().mean())).round(1).to_string())

### Cobertura do conjunto de teste reservado

Cabe registrar uma limitação da partição, revelada por esta análise. Os oito ensaios de placa de plástico distribuem-se por três dobras da validação, e nenhum deles caiu no conjunto de teste reservado, que reúne apenas ensaios de suporte de bobina em alumínio.

A estratificação equilibrou a contagem de ensaios e a distribuição de casos de ensaio, e não o material do suporte. Com oito ensaios de plástico em cinquenta e seis, e sob agrupamento por configuração, o resultado ocorre sem que nenhuma regra tenha sido violada.

A consequência delimita o que a avaliação final autoriza afirmar: o desempenho medido sobre o teste reservado vale para o regime de alumínio, e não estima o desempenho sobre placas de plástico. A alternativa de refazer a partição para incluir plástico no teste foi considerada e descartada, uma vez que retiraria do treino metade dos ensaios do regime mais difícil, que já é o de menor densidade. A limitação é, portanto, assumida e declarada, e não corrigida.

## 19. Modelo por área, formulação de referência

Esta seção estabelece a formulação de referência, com floresta aleatória em todas as regiões. Ela não é a formulação entregue da quarta sprint, que consta da seção 23 e escolhe o estimador por região. As seções 19 e 20 permanecem porque é contra elas que os ganhos posteriores são medidos.

A solução adotada é composta por sete modelos, um por área térmica, em lugar dos setenta e quatro modelos por componente das seções anteriores. O algoritmo é o Random Forest e a base é a de treze categorias.

### O que muda na formulação

Cada área passa a ter um único modelo, treinado sobre o formato longo daquela área, em que cada linha é um par de ensaio e componente. Ao vetor de configuração somam-se dois atributos que descrevem o componente, ambos calculados exclusivamente sobre as linhas de treino de cada dobra: o patamar térmico, que é a média do componente, e a escala, que é o seu desvio-padrão.

Esses dois atributos são o que torna a formulação viável. Sem eles, o modelo precisa aprender o patamar de cada componente a partir de colunas indicadoras, e gasta nisso a capacidade que deveria usar para aprender o efeito da configuração: essa variante foi medida e resultou em erro de 14,02 graus, o dobro do obtido aqui. Com o patamar sozinho, o erro cai para 7,17. Acrescentar a escala reconhece que componentes não diferem apenas em temperatura média, mas também na sensibilidade com que respondem à configuração.

Cabe registrar que a alternativa de subtrair o patamar do alvo, em vez de fornecê-lo como entrada, foi testada e é pior: ela impõe um único mecanismo de resposta compartilhado por todos os componentes da área, e eles não respondem da mesma forma.

### O que a mudança custa e o que ela traz

Sobre as mesmas dobras e a mesma base, a formulação por área apresenta erro absoluto médio de 6,57 graus, contra 6,92 dos setenta e quatro modelos. A comparação pareada por componente devolve diferença de menos 0,379 grau em favor da formulação por área, com p de 0,0097 no teste t pareado e intervalo de confiança de menos 0,664 a menos 0,095, que não contém zero. A formulação por área melhora em 44 dos 74 componentes.

O ganho aparece também no coeficiente de determinação por componente, que passa de menos 0,080 para mais 0,164, com 23 componentes negativos contra 29. É a primeira formulação medida no projeto cujo coeficiente médio por componente é positivo, ou seja, que supera de forma consistente a previsão pela média histórica do próprio componente.

Registra-se também um ganho de natureza prática. Sete modelos constituem um artefato treinável e versionável, ao passo que setenta e quatro tornam a persistência do modelo trabalhosa. O repositório não salvava modelo algum até aqui, e foi por isso que a tela de triagem precisou de uma exportação de grade de previsões em vez de carregar pesos.

### Limitação da evidência

O resultado não foi confirmado em dado independente. O conjunto de teste reservado foi utilizado uma única vez, em 11 de setembro de 2026, para comparar duas outras formulações, e não está mais disponível como árbitro imparcial. Registra-se ainda que diversas variantes desta formulação foram comparadas sobre as mesmas dobras, de modo que parte da vantagem observada pode decorrer da própria seleção. A confirmação depende de ensaios adicionais, cujo pedido consta do item a desta seção na documentação.

Registra-se, por fim, uma contrapartida medida. O Gradient Boosting com 300 árvores e taxa de aprendizado de 0,05, sob esta mesma formulação, apresenta erro maior, de 7,06 graus, e deixa passar menos excessos: 26 na margem de 8 graus, contra 36 do Random Forest. O algoritmo que minimiza o erro médio não é o que minimiza a aprovação indevida, e a escolha entre os dois é uma troca entre precisão e segurança, e não uma questão de qual é melhor.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Um modelo por area, sobre o formato longo. Os dois atributos de componente
# sao calculados apenas sobre as linhas de treino de cada dobra.
#
# O numero de arvores e fixado em 300 e os demais parametros permanecem no
# padrao. Cabe registrar que max_features nao e alterado: o padrao de regressao
# considera todas as colunas em cada divisao, e a alternativa da raiz do total,
# que e o padrao de classificacao, foi medida e custa 1,55 grau no agregado, por
# deixar de fora as colunas dominantes na maioria das divisoes.
AREA_PARAMS = dict(n_estimators=300, random_state=42, n_jobs=-1)


def formato_longo(nome, df):
    """Uma linha por par de ensaio e componente medido."""
    X, cols_alvo = separar(df)
    # As colunas de ordem de acionamento ficam fora desta secao; elas entram
    # apenas na Secao 21, de modo que a comparacao daquela secao isole o efeito
    # da codificacao e nao o da inclusao dessas colunas.
    X = X.drop(columns=[c for c in X.columns if c.startswith(ORDEM_ACIONAMENTO)])
    # Trava contra filtro que deixa de casar: um renome de coluna ja fez este
    # descarte parar de encontrar a ordem de acionamento, e as quatro colunas
    # entraram em silencio, piorando o resultado sem erro algum.
    # A verificacao usa o mesmo prefixo do descarte. Procurar a palavra solta
    # produzia falso positivo, porque os rotulos de perda das treze categorias a
    # contem no proprio texto, como em losses_broil c2 simone - front order.
    _restou = [c for c in X.columns if c.startswith(ORDEM_ACIONAMENTO)]
    assert not _restou, f"ordem de acionamento nao foi descartada: {_restou}"
    blocos = []
    for col in cols_alvo:
        y = df[col]
        if y.notna().sum() < MIN_PREDICOES:
            continue
        sub = X[y.notna()].copy()
        sub["_run"] = (df["run_idx"][y.notna()].to_numpy()
                       if "run_idx" in df.columns else np.arange(len(df))[y.notna()])
        sub["_componente"] = col
        sub["_alvo"] = y[y.notna()].to_numpy()
        blocos.append(sub)
    longo = pd.concat(blocos, ignore_index=True)
    longo["_dobra"] = longo["_run"].map(DOBRA)
    return longo[longo["_dobra"].notna()].reset_index(drop=True)


def atributos_do_componente(longo, treino):
    """Patamar e escala de cada componente, medidos so no treino da dobra."""
    g = longo[treino].groupby("_componente")["_alvo"]
    media = g.mean()
    desvio = g.std().fillna(1.0).replace(0, 1.0)
    geral = float(longo.loc[treino, "_alvo"].mean())
    return media, desvio, geral


def matriz(longo, mascara, entradas, media, desvio, geral):
    comp = longo.loc[mascara, "_componente"]
    return np.hstack([
        longo.loc[mascara, entradas].to_numpy(),
        comp.map(media).fillna(geral).to_numpy().reshape(-1, 1),
        comp.map(desvio).fillna(1.0).to_numpy().reshape(-1, 1),
    ])


predicoes_area = []
for nome, df in sorted(datasets.items()):
    longo = formato_longo(nome, df)
    entradas = [c for c in longo.columns if not c.startswith("_")]
    for d in DOBRAS:
        treino = (longo["_dobra"] != d).to_numpy()
        val = (longo["_dobra"] == d).to_numpy()
        if treino.sum() < MIN_TREINO or not val.any():
            continue
        media, desvio, geral = atributos_do_componente(longo, treino)
        X_tr = matriz(longo, treino, entradas, media, desvio, geral)
        X_val = matriz(longo, val, entradas, media, desvio, geral)
        escala = StandardScaler()
        modelo = RandomForestRegressor(**AREA_PARAMS)
        modelo.fit(escala.fit_transform(X_tr), longo.loc[treino, "_alvo"])
        p = modelo.predict(escala.transform(X_val))
        bloco = longo.loc[val, ["_componente", "_run", "_alvo"]].copy()
        bloco.columns = ["componente", "ensaio", "real"]
        bloco["dataset"] = nome
        bloco["previsto"] = p
        predicoes_area.append(bloco)

predicoes_area = pd.concat(predicoes_area, ignore_index=True)
predicoes_area["erro"] = predicoes_area["previsto"] - predicoes_area["real"]
predicoes_area["limite"] = predicoes_area["componente"].map(limite_de)

print(f"Modelos treinados: {len(datasets)}, um por area")
print(f"Componentes cobertos: {predicoes_area['componente'].nunique()}")
print(f"Predicoes de validacao: {len(predicoes_area)}")
print()

resultados_area = []
for nome, g in predicoes_area.groupby("dataset"):
    resultados_area.append({
        "dataset": nome, "componentes": g["componente"].nunique(),
        **avaliar(g["real"].to_numpy(), g["previsto"].to_numpy()),
    })
df_area = pd.DataFrame(resultados_area)
print("Desempenho por area, modelo unico da area:")
print(df_area[["dataset", "componentes", "n", "MAE", "RMSE", "R2", "vies",
               "MAPE_%", "dentro_10_%"]].round(2).to_string(index=False))
print()

e = predicoes_area["erro"]
rel = e.abs() / predicoes_area["real"]
print(f"AGREGADO: MAE {e.abs().mean():.2f} C | RMSE {np.sqrt((e ** 2).mean()):.2f} C | "
      f"MAPE {100 * rel.mean():.2f}% | dentro de 10%: {100 * (rel <= 0.10).mean():.1f}%")
print()

# Calibracao da margem de seguranca sobre a formulacao adotada.
com_limite = predicoes_area[predicoes_area["limite"].notna()].copy()
estoura = com_limite["real"] >= com_limite["limite"]
print(f"Leituras que atingem o limite: {int(estoura.sum())} de {len(com_limite)}")
print(f"Vies onde nao estoura: {com_limite[~estoura]['erro'].mean():+.2f} C | "
      f"onde estoura: {com_limite[estoura]['erro'].mean():+.2f} C")
print()
print(f"{'margem':>7} {'falsos aprovados':>18} {'falsos reprovados':>19} {'deteccao':>10}")
for margem in (0, 3, 5, 8, 10, 12):
    sinal = (com_limite["previsto"] + margem) >= com_limite["limite"]
    fa = int((estoura & ~sinal).sum())
    fr = int((~estoura & sinal).sum())
    print(f"{margem:>7} {fa:>18} {fr:>19} {100 * (estoura.sum() - fa) / estoura.sum():>9.1f}%")

### Comparação direta com a formulação por componente

A célula abaixo confronta as duas formulações sobre as mesmas dobras e os mesmos componentes, de modo que a decisão fique documentada com número, e não apenas com texto.

In [ ]:
from scipy import stats

# A formulacao por componente vem da Secao 13, ja calculada.
por_comp = []
for (area, col), (reais, preditos, limite, ensaios) in predicoes_gb.items():
    for real, previsto, ensaio in zip(reais, preditos, ensaios):
        por_comp.append({"dataset": area, "componente": col, "ensaio": int(ensaio),
                         "real": real, "previsto": previsto, "limite": limite})
por_comp = pd.DataFrame(por_comp)
por_comp["erro"] = por_comp["previsto"] - por_comp["real"]


def _mae(t):
    return t.assign(e=t["erro"].abs()).groupby("componente")["e"].mean()


a, b = _mae(por_comp), _mae(predicoes_area)
comuns = sorted(set(a.index) & set(b.index))
x, y = a[comuns].to_numpy(), b[comuns].to_numpy()
dif = y - x
ic = stats.t.interval(0.95, len(dif) - 1, loc=dif.mean(), scale=stats.sem(dif))
print(f"Componentes comparados: {len(comuns)}")
print(f"MAE macro | por componente {x.mean():.3f} C | por area {y.mean():.3f} C")
print(f"Diferenca {dif.mean():+.3f} C | a formulacao por area melhora em "
      f"{int((dif < 0).sum())} componentes")
print(f"Teste t pareado p={stats.ttest_rel(y, x).pvalue:.3f} | "
      f"IC 95% [{ic[0]:+.3f}, {ic[1]:+.3f}] C")
print()

# Decisao: o desfecho e binario e medido no mesmo par, de modo que o teste
# proprio e o de McNemar sobre os pares em que as formulacoes discordam.
MARGEM = MARGEM_DECISAO


def _decide(t):
    t = t[t["limite"].notna()].set_index(["componente", "ensaio"])
    return (t["real"] >= t["limite"]), ((t["previsto"] + MARGEM) >= t["limite"])


est_a, sin_a = _decide(por_comp)
est_b, sin_b = _decide(predicoes_area)
idx = est_a.index.intersection(est_b.index)
est = est_a[idx]
so_comp = int((est & sin_a[idx] & ~sin_b[idx]).sum())
so_area = int((est & sin_b[idx] & ~sin_a[idx]).sum())
print(f"Excessos reais avaliados: {int(est.sum())}")
print(f"Falsos aprovados | por componente {int((est & ~sin_a[idx]).sum())} | "
      f"por area {int((est & ~sin_b[idx]).sum())}")
print(f"So a formulacao por componente detecta: {so_comp}")
print(f"So a formulacao por area detecta:       {so_area}")
if so_comp + so_area:
    p = stats.binomtest(so_comp, so_comp + so_area, 0.5).pvalue
    print(f"McNemar exato sobre {so_comp + so_area} pares discordantes: p={p:.4f}")

## 20. Avaliação da formulação de referência no conjunto de teste reservado

As seções anteriores reportam desempenho medido por validação cruzada dentro do conjunto de treino. Esta seção reporta o desempenho sobre os 10 ensaios reservados, que não participaram de ajuste algum: nem do treino dos modelos, nem da escolha de hiperparâmetros, nem da seleção entre formulações.

Os sete modelos são retreinados aqui com a totalidade dos 46 ensaios de treino, que é a condição do modelo efetivamente entregue, e aplicados uma única vez ao conjunto reservado.

### O que esta avaliação autoriza afirmar, e o que não autoriza

Os números desta seção são melhores que os da validação, e a diferença não deve ser lida como ganho de desempenho. Ela decorre da composição do conjunto reservado, descrita no item a da Seção 4.3 da documentação: os 10 ensaios são todos de suporte de bobina em alumínio e nenhum é do modo broil c1. O regime de maior temperatura, em que o modelo mais erra, está ausente.

A leitura conservadora é, portanto, a da validação cruzada, que cobre os dois regimes. A leitura desta seção vale para o regime de alumínio.

Registra-se ainda que o conjunto reservado foi consultado uma vez antes desta avaliação, em 11 de setembro de 2026, para decidir entre duas formulações que não são a adotada. A consulta está registrada no item c da documentação, e por causa dela o conjunto deixou de ser árbitro plenamente independente para escolhas futuras.

In [ ]:
# Os sete modelos sao retreinados com todo o conjunto de treino e aplicados
# uma unica vez aos ensaios reservados.
IDX_TREINO = set(particao["run_idx_treino"])
IDX_TESTE = set(particao["run_idx_teste"])
print(f"Treino: {len(IDX_TREINO)} ensaios | teste reservado: {len(IDX_TESTE)} ensaios")
print()

def formato_longo_completo(df):
    """Como formato_longo, mas conserva os ensaios de teste.

    A funcao da Secao 19 descarta as linhas sem dobra atribuida, que sao
    justamente as do conjunto reservado; aqui elas precisam ser mantidas.
    """
    X, cols_alvo = separar(df)
    # As colunas de ordem de acionamento ficam fora desta secao; elas entram
    # apenas na Secao 21, de modo que a comparacao daquela secao isole o efeito
    # da codificacao e nao o da inclusao dessas colunas.
    X = X.drop(columns=[c for c in X.columns if c.startswith(ORDEM_ACIONAMENTO)])
    # Trava contra filtro que deixa de casar: um renome de coluna ja fez este
    # descarte parar de encontrar a ordem de acionamento, e as quatro colunas
    # entraram em silencio, piorando o resultado sem erro algum.
    # A verificacao usa o mesmo prefixo do descarte. Procurar a palavra solta
    # produzia falso positivo, porque os rotulos de perda das treze categorias a
    # contem no proprio texto, como em losses_broil c2 simone - front order.
    _restou = [c for c in X.columns if c.startswith(ORDEM_ACIONAMENTO)]
    assert not _restou, f"ordem de acionamento nao foi descartada: {_restou}"
    blocos = []
    for col in cols_alvo:
        y = df[col]
        if y.notna().sum() < MIN_PREDICOES:
            continue
        sub = X[y.notna()].copy()
        sub["_run"] = (df["run_idx"][y.notna()].to_numpy()
                       if "run_idx" in df.columns else np.arange(len(df))[y.notna()])
        sub["_componente"] = col
        sub["_alvo"] = y[y.notna()].to_numpy()
        blocos.append(sub)
    return pd.concat(blocos, ignore_index=True)


predicoes_teste = []
for nome, df in sorted(datasets.items()):
    longo = formato_longo_completo(df)
    entradas = [c for c in longo.columns if not c.startswith("_")]
    treino = longo["_run"].isin(IDX_TREINO).to_numpy()
    teste = longo["_run"].isin(IDX_TESTE).to_numpy()
    if treino.sum() < MIN_TREINO or not teste.any():
        continue
    media, desvio, geral = atributos_do_componente(longo, treino)
    escala = StandardScaler()
    modelo = RandomForestRegressor(**AREA_PARAMS)
    modelo.fit(escala.fit_transform(matriz(longo, treino, entradas, media, desvio, geral)),
               longo.loc[treino, "_alvo"])
    p = modelo.predict(escala.transform(
        matriz(longo, teste, entradas, media, desvio, geral)))
    bloco = longo.loc[teste, ["_componente", "_run", "_alvo"]].copy()
    bloco.columns = ["componente", "ensaio", "real"]
    bloco["dataset"] = nome
    bloco["previsto"] = p
    predicoes_teste.append(bloco)

predicoes_teste = pd.concat(predicoes_teste, ignore_index=True)
predicoes_teste["erro"] = predicoes_teste["previsto"] - predicoes_teste["real"]
predicoes_teste["limite"] = predicoes_teste["componente"].map(limite_de)

resultados_teste = []
for nome, g in predicoes_teste.groupby("dataset"):
    resultados_teste.append({
        "dataset": nome, "componentes": g["componente"].nunique(),
        **avaliar(g["real"].to_numpy(), g["previsto"].to_numpy()),
    })
df_teste = pd.DataFrame(resultados_teste)
print("Desempenho no conjunto de teste reservado, por area:")
print(df_teste[["dataset", "componentes", "n", "MAE", "RMSE", "R2", "vies",
                "MAPE_%", "dentro_10_%"]].round(2).to_string(index=False))
print()

e = predicoes_teste["erro"]
rel = e.abs() / predicoes_teste["real"]
print(f"AGREGADO no teste: MAE {e.abs().mean():.2f} C | RMSE {np.sqrt((e ** 2).mean()):.2f} C | "
      f"MAPE {100 * rel.mean():.2f}% | dentro de 10%: {100 * (rel <= 0.10).mean():.1f}% | "
      f"vies {e.mean():+.2f} C")
print(f"Na validacao cruzada, para comparacao: MAE {predicoes_area['erro'].abs().mean():.2f} C")
print()

com_lim = predicoes_teste[predicoes_teste["limite"].notna()].copy()
estoura = com_lim["real"] >= com_lim["limite"]
print(f"Leituras com limite: {len(com_lim)} | que atingem o limite: {int(estoura.sum())}")
print(f"Vies onde nao atinge: {com_lim[~estoura]['erro'].mean():+.2f} C | "
      f"onde atinge: {com_lim[estoura]['erro'].mean():+.2f} C")
print()
print(f"{'margem':>7} {'falsos aprovados':>18} {'falsos reprovados':>19} {'deteccao':>10}")
for margem in (0, 5, 8, 10, 12):
    sinal = (com_lim["previsto"] + margem) >= com_lim["limite"]
    fa = int((estoura & ~sinal).sum())
    fr = int((~estoura & sinal).sum())
    print(f"{margem:>7} {fa:>18} {fr:>19} {100 * (estoura.sum() - fa) / estoura.sum():>9.1f}%")

## 21. Formulação melhorada: codificação, estimador e escala do alvo escolhidos por região

A investigação de melhoria do conjunto de dados, registrada em notebooks/colabs_S3/analise_dataset, mediu vinte e cinco possibilidades sob o protocolo deste projeto. Três apresentaram ganho reprodutível, e as três reorganizam informação já presente na base, sem criar dado algum.

A primeira é a codificação da coluna de perdas escolhida por região, entre três alternativas: o perfil de perdas da entrega; o perfil acrescido da ordem de acionamento dos IGBTs, extraída na preparação; e as treze categorias originais, que na região das bobinas carregam informação sobre a potência aplicada a cada uma. A segunda é o estimador escolhido por região, entre o Random Forest e a média dele com o Gradient Boosting. A terceira é a escala do alvo, entre a original e o logaritmo, que na região das bobinas reduz o erro em mais de um grau por corrigir a estrutura multiplicativa do erro em temperatura alta.

Toda escolha é feita dentro de cada dobra, usando apenas as demais dobras, de modo que a dobra avaliada nunca participa da decisão que a afeta. O custo de escolher é, portanto, pago no número reportado. O procedimento é repetido em três sementes.

O conjunto de teste reservado não é consultado nesta seção. A Seção 20 permanece como a única avaliação no teste, e refere-se ao modelo entregue.

In [ ]:
from itertools import product
from sklearn.ensemble import GradientBoostingRegressor

SEMENTES_21 = (42, 1, 2)
ENCODINGS = ("perfil", "perfil+ordem", "13cat")
ESTIMADORES = ("rf", "rf+gb")
ESCALAS = ("linear", "log")


def entradas_por_codificacao(nome, _df_ignorado=None):
    """Devolve, para cada codificacao, a tabela de entradas alinhada por ensaio.

    Cada bloco vem de onde de fato pertence. O perfil de perdas e lido da pasta
    da terceira sprint, que e o registro daquela entrega. A ordem de chaveamento
    e uma construcao da quarta sprint e vem da pasta desta. As treze categorias
    tambem, sem a ordem, para que a comparacao entre codificacoes isole o efeito
    da codificacao das perdas.

    Duas armadilhas ja capturadas justificam a verificacao ao final. Derivar o
    perfil do conjunto carregado com SUFIXO fazia com que ele coincidisse com as
    treze categorias. E buscar a ordem na pasta da terceira sprint, depois de ela
    ser regenerada sem essas colunas, fazia perfil e perfil com ordem coincidirem.
    Nos dois casos a grade comparava copias da mesma tabela sem que nada falhasse.
    """
    df_perfil = pd.read_csv(resolve(PASTA_DADOS_PERFIL, f"dataset_{nome}_perfil.csv"))
    X_p, _ = separar(df_perfil)
    X_p = X_p[[c for c in X_p.columns if not c.startswith(ORDEM_ACIONAMENTO)]]

    df13 = pd.read_csv(resolve(PASTA_DADOS, f"dataset_{nome}_13cat.csv"))
    X_13_completo, _ = separar(df13)
    ordem = [c for c in X_13_completo.columns if c.startswith(ORDEM_ACIONAMENTO)]
    X_13 = X_13_completo[[c for c in X_13_completo.columns if c not in ordem]]

    saida = {
        "perfil": X_p,
        "perfil+ordem": pd.concat([X_p, X_13_completo[ordem]], axis=1),
        "13cat": X_13,
    }
    # As tres codificacoes precisam ser efetivamente distintas, sob pena de a
    # grade medir a mesma coisa tres vezes e o resultado parecer legitimo.
    assert len(ordem) > 0, f"{nome}: nenhuma coluna de ordem de chaveamento encontrada"
    assinaturas = {k: tuple(sorted(v.columns)) for k, v in saida.items()}
    assert len(set(assinaturas.values())) == 3, \
        f"{nome}: codificacoes coincidem entre si: { {k: len(v) for k, v in assinaturas.items()} }"
    return saida


def longo_de(df, X):
    """Formato longo com as entradas fornecidas, so ensaios com dobra."""
    _, cols_alvo = separar(df)
    blocos = []
    for col in cols_alvo:
        y = df[col]
        if y.notna().sum() < MIN_PREDICOES:
            continue
        sub = X[y.notna()].copy()
        sub["_run"] = np.arange(len(df))[y.notna()]
        sub["_componente"] = col
        sub["_alvo"] = y[y.notna()].to_numpy()
        blocos.append(sub)
    longo = pd.concat(blocos, ignore_index=True)
    longo["_dobra"] = longo["_run"].map(DOBRA)
    return longo[longo["_dobra"].notna()].reset_index(drop=True)



# erro[(area, enc, est, escala, semente)] = {(dobra, componente, run): erro absoluto}
# A grade e avaliada em uma celula por regiao, para que nenhuma celula ultrapasse
# o limite de execucao; o dicionario e compartilhado entre elas.
erro = {}
print("configuracao da grade: regioes", sorted(datasets), "| sementes", SEMENTES_21)

In [ ]:
REGIAO_21 = "oven_hmi"
# erro[(area, enc, est, escala, semente)] = {(dobra, componente, run): erro absoluto}
for nome, df in [(REGIAO_21, datasets[REGIAO_21])]:
    Xs = entradas_por_codificacao(nome, df)
    for enc in ENCODINGS:
        longo = longo_de(df, Xs[enc])
        entradas = [c for c in longo.columns if not c.startswith("_")]
        for escala_alvo in ESCALAS:
            alvo = np.log(longo["_alvo"].to_numpy()) if escala_alvo == "log" else longo["_alvo"].to_numpy()
            gb_cache = {}
            for d in DOBRAS:
                treino = (longo["_dobra"] != d).to_numpy()
                val = (longo["_dobra"] == d).to_numpy()
                if treino.sum() < MIN_TREINO or not val.any():
                    continue
                # patamar e escala do componente, na escala do alvo em uso
                g = pd.Series(alvo[treino]).groupby(longo.loc[treino, "_componente"].to_numpy())
                media, desvio = g.mean(), g.std().fillna(1.0).replace(0, 1.0)
                geral = float(alvo[treino].mean())
                X_tr = matriz(longo, treino, entradas, media, desvio, geral)
                X_val = matriz(longo, val, entradas, media, desvio, geral)
                esc = StandardScaler()
                X_tr, X_val = esc.fit_transform(X_tr), esc.transform(X_val)
                # o Gradient Boosting e deterministico; um ajuste serve a todas as sementes
                gb_cache[d] = GradientBoostingRegressor(n_estimators=300, random_state=0).fit(
                    X_tr, alvo[treino]).predict(X_val)
                chaves = [(d, longo.at[i, "_componente"], int(longo.at[i, "_run"]))
                          for i in np.where(val)[0]]
                reais = longo.loc[val, "_alvo"].to_numpy()
                for semente in SEMENTES_21:
                    p_rf = RandomForestRegressor(n_estimators=300, random_state=semente,
                                                 n_jobs=-1).fit(X_tr, alvo[treino]).predict(X_val)
                    for est in ESTIMADORES:
                        p = p_rf if est == "rf" else (p_rf + gb_cache[d]) / 2
                        if escala_alvo == "log":
                            p = np.exp(p)
                        erro.setdefault((nome, enc, est, escala_alvo, semente), {}).update(
                            {ch: abs(p[i] - reais[i]) for i, ch in enumerate(chaves)})
    print(f"  {nome}: concluida", flush=True)
print("grade completa avaliada")

In [ ]:
REGIAO_21 = "ct_knobs"
# erro[(area, enc, est, escala, semente)] = {(dobra, componente, run): erro absoluto}
for nome, df in [(REGIAO_21, datasets[REGIAO_21])]:
    Xs = entradas_por_codificacao(nome, df)
    for enc in ENCODINGS:
        longo = longo_de(df, Xs[enc])
        entradas = [c for c in longo.columns if not c.startswith("_")]
        for escala_alvo in ESCALAS:
            alvo = np.log(longo["_alvo"].to_numpy()) if escala_alvo == "log" else longo["_alvo"].to_numpy()
            gb_cache = {}
            for d in DOBRAS:
                treino = (longo["_dobra"] != d).to_numpy()
                val = (longo["_dobra"] == d).to_numpy()
                if treino.sum() < MIN_TREINO or not val.any():
                    continue
                # patamar e escala do componente, na escala do alvo em uso
                g = pd.Series(alvo[treino]).groupby(longo.loc[treino, "_componente"].to_numpy())
                media, desvio = g.mean(), g.std().fillna(1.0).replace(0, 1.0)
                geral = float(alvo[treino].mean())
                X_tr = matriz(longo, treino, entradas, media, desvio, geral)
                X_val = matriz(longo, val, entradas, media, desvio, geral)
                esc = StandardScaler()
                X_tr, X_val = esc.fit_transform(X_tr), esc.transform(X_val)
                # o Gradient Boosting e deterministico; um ajuste serve a todas as sementes
                gb_cache[d] = GradientBoostingRegressor(n_estimators=300, random_state=0).fit(
                    X_tr, alvo[treino]).predict(X_val)
                chaves = [(d, longo.at[i, "_componente"], int(longo.at[i, "_run"]))
                          for i in np.where(val)[0]]
                reais = longo.loc[val, "_alvo"].to_numpy()
                for semente in SEMENTES_21:
                    p_rf = RandomForestRegressor(n_estimators=300, random_state=semente,
                                                 n_jobs=-1).fit(X_tr, alvo[treino]).predict(X_val)
                    for est in ESTIMADORES:
                        p = p_rf if est == "rf" else (p_rf + gb_cache[d]) / 2
                        if escala_alvo == "log":
                            p = np.exp(p)
                        erro.setdefault((nome, enc, est, escala_alvo, semente), {}).update(
                            {ch: abs(p[i] - reais[i]) for i, ch in enumerate(chaves)})
    print(f"  {nome}: concluida", flush=True)
print("grade completa avaliada")

In [ ]:
REGIAO_21 = "ipc_cooling"
# erro[(area, enc, est, escala, semente)] = {(dobra, componente, run): erro absoluto}
for nome, df in [(REGIAO_21, datasets[REGIAO_21])]:
    Xs = entradas_por_codificacao(nome, df)
    for enc in ENCODINGS:
        longo = longo_de(df, Xs[enc])
        entradas = [c for c in longo.columns if not c.startswith("_")]
        for escala_alvo in ESCALAS:
            alvo = np.log(longo["_alvo"].to_numpy()) if escala_alvo == "log" else longo["_alvo"].to_numpy()
            gb_cache = {}
            for d in DOBRAS:
                treino = (longo["_dobra"] != d).to_numpy()
                val = (longo["_dobra"] == d).to_numpy()
                if treino.sum() < MIN_TREINO or not val.any():
                    continue
                # patamar e escala do componente, na escala do alvo em uso
                g = pd.Series(alvo[treino]).groupby(longo.loc[treino, "_componente"].to_numpy())
                media, desvio = g.mean(), g.std().fillna(1.0).replace(0, 1.0)
                geral = float(alvo[treino].mean())
                X_tr = matriz(longo, treino, entradas, media, desvio, geral)
                X_val = matriz(longo, val, entradas, media, desvio, geral)
                esc = StandardScaler()
                X_tr, X_val = esc.fit_transform(X_tr), esc.transform(X_val)
                # o Gradient Boosting e deterministico; um ajuste serve a todas as sementes
                gb_cache[d] = GradientBoostingRegressor(n_estimators=300, random_state=0).fit(
                    X_tr, alvo[treino]).predict(X_val)
                chaves = [(d, longo.at[i, "_componente"], int(longo.at[i, "_run"]))
                          for i in np.where(val)[0]]
                reais = longo.loc[val, "_alvo"].to_numpy()
                for semente in SEMENTES_21:
                    p_rf = RandomForestRegressor(n_estimators=300, random_state=semente,
                                                 n_jobs=-1).fit(X_tr, alvo[treino]).predict(X_val)
                    for est in ESTIMADORES:
                        p = p_rf if est == "rf" else (p_rf + gb_cache[d]) / 2
                        if escala_alvo == "log":
                            p = np.exp(p)
                        erro.setdefault((nome, enc, est, escala_alvo, semente), {}).update(
                            {ch: abs(p[i] - reais[i]) for i, ch in enumerate(chaves)})
    print(f"  {nome}: concluida", flush=True)
print("grade completa avaliada")

In [ ]:
REGIAO_21 = "main_vent_sys"
# erro[(area, enc, est, escala, semente)] = {(dobra, componente, run): erro absoluto}
for nome, df in [(REGIAO_21, datasets[REGIAO_21])]:
    Xs = entradas_por_codificacao(nome, df)
    for enc in ENCODINGS:
        longo = longo_de(df, Xs[enc])
        entradas = [c for c in longo.columns if not c.startswith("_")]
        for escala_alvo in ESCALAS:
            alvo = np.log(longo["_alvo"].to_numpy()) if escala_alvo == "log" else longo["_alvo"].to_numpy()
            gb_cache = {}
            for d in DOBRAS:
                treino = (longo["_dobra"] != d).to_numpy()
                val = (longo["_dobra"] == d).to_numpy()
                if treino.sum() < MIN_TREINO or not val.any():
                    continue
                # patamar e escala do componente, na escala do alvo em uso
                g = pd.Series(alvo[treino]).groupby(longo.loc[treino, "_componente"].to_numpy())
                media, desvio = g.mean(), g.std().fillna(1.0).replace(0, 1.0)
                geral = float(alvo[treino].mean())
                X_tr = matriz(longo, treino, entradas, media, desvio, geral)
                X_val = matriz(longo, val, entradas, media, desvio, geral)
                esc = StandardScaler()
                X_tr, X_val = esc.fit_transform(X_tr), esc.transform(X_val)
                # o Gradient Boosting e deterministico; um ajuste serve a todas as sementes
                gb_cache[d] = GradientBoostingRegressor(n_estimators=300, random_state=0).fit(
                    X_tr, alvo[treino]).predict(X_val)
                chaves = [(d, longo.at[i, "_componente"], int(longo.at[i, "_run"]))
                          for i in np.where(val)[0]]
                reais = longo.loc[val, "_alvo"].to_numpy()
                for semente in SEMENTES_21:
                    p_rf = RandomForestRegressor(n_estimators=300, random_state=semente,
                                                 n_jobs=-1).fit(X_tr, alvo[treino]).predict(X_val)
                    for est in ESTIMADORES:
                        p = p_rf if est == "rf" else (p_rf + gb_cache[d]) / 2
                        if escala_alvo == "log":
                            p = np.exp(p)
                        erro.setdefault((nome, enc, est, escala_alvo, semente), {}).update(
                            {ch: abs(p[i] - reais[i]) for i, ch in enumerate(chaves)})
    print(f"  {nome}: concluida", flush=True)
print("grade completa avaliada")

In [ ]:
REGIAO_21 = "mainback"
# erro[(area, enc, est, escala, semente)] = {(dobra, componente, run): erro absoluto}
for nome, df in [(REGIAO_21, datasets[REGIAO_21])]:
    Xs = entradas_por_codificacao(nome, df)
    for enc in ENCODINGS:
        longo = longo_de(df, Xs[enc])
        entradas = [c for c in longo.columns if not c.startswith("_")]
        for escala_alvo in ESCALAS:
            alvo = np.log(longo["_alvo"].to_numpy()) if escala_alvo == "log" else longo["_alvo"].to_numpy()
            gb_cache = {}
            for d in DOBRAS:
                treino = (longo["_dobra"] != d).to_numpy()
                val = (longo["_dobra"] == d).to_numpy()
                if treino.sum() < MIN_TREINO or not val.any():
                    continue
                # patamar e escala do componente, na escala do alvo em uso
                g = pd.Series(alvo[treino]).groupby(longo.loc[treino, "_componente"].to_numpy())
                media, desvio = g.mean(), g.std().fillna(1.0).replace(0, 1.0)
                geral = float(alvo[treino].mean())
                X_tr = matriz(longo, treino, entradas, media, desvio, geral)
                X_val = matriz(longo, val, entradas, media, desvio, geral)
                esc = StandardScaler()
                X_tr, X_val = esc.fit_transform(X_tr), esc.transform(X_val)
                # o Gradient Boosting e deterministico; um ajuste serve a todas as sementes
                gb_cache[d] = GradientBoostingRegressor(n_estimators=300, random_state=0).fit(
                    X_tr, alvo[treino]).predict(X_val)
                chaves = [(d, longo.at[i, "_componente"], int(longo.at[i, "_run"]))
                          for i in np.where(val)[0]]
                reais = longo.loc[val, "_alvo"].to_numpy()
                for semente in SEMENTES_21:
                    p_rf = RandomForestRegressor(n_estimators=300, random_state=semente,
                                                 n_jobs=-1).fit(X_tr, alvo[treino]).predict(X_val)
                    for est in ESTIMADORES:
                        p = p_rf if est == "rf" else (p_rf + gb_cache[d]) / 2
                        if escala_alvo == "log":
                            p = np.exp(p)
                        erro.setdefault((nome, enc, est, escala_alvo, semente), {}).update(
                            {ch: abs(p[i] - reais[i]) for i, ch in enumerate(chaves)})
    print(f"  {nome}: concluida", flush=True)
print("grade completa avaliada")

In [ ]:
REGIAO_21 = "coil_temps"
# erro[(area, enc, est, escala, semente)] = {(dobra, componente, run): erro absoluto}
for nome, df in [(REGIAO_21, datasets[REGIAO_21])]:
    Xs = entradas_por_codificacao(nome, df)
    for enc in ENCODINGS:
        longo = longo_de(df, Xs[enc])
        entradas = [c for c in longo.columns if not c.startswith("_")]
        for escala_alvo in ESCALAS:
            alvo = np.log(longo["_alvo"].to_numpy()) if escala_alvo == "log" else longo["_alvo"].to_numpy()
            gb_cache = {}
            for d in DOBRAS:
                treino = (longo["_dobra"] != d).to_numpy()
                val = (longo["_dobra"] == d).to_numpy()
                if treino.sum() < MIN_TREINO or not val.any():
                    continue
                # patamar e escala do componente, na escala do alvo em uso
                g = pd.Series(alvo[treino]).groupby(longo.loc[treino, "_componente"].to_numpy())
                media, desvio = g.mean(), g.std().fillna(1.0).replace(0, 1.0)
                geral = float(alvo[treino].mean())
                X_tr = matriz(longo, treino, entradas, media, desvio, geral)
                X_val = matriz(longo, val, entradas, media, desvio, geral)
                esc = StandardScaler()
                X_tr, X_val = esc.fit_transform(X_tr), esc.transform(X_val)
                # o Gradient Boosting e deterministico; um ajuste serve a todas as sementes
                gb_cache[d] = GradientBoostingRegressor(n_estimators=300, random_state=0).fit(
                    X_tr, alvo[treino]).predict(X_val)
                chaves = [(d, longo.at[i, "_componente"], int(longo.at[i, "_run"]))
                          for i in np.where(val)[0]]
                reais = longo.loc[val, "_alvo"].to_numpy()
                for semente in SEMENTES_21:
                    p_rf = RandomForestRegressor(n_estimators=300, random_state=semente,
                                                 n_jobs=-1).fit(X_tr, alvo[treino]).predict(X_val)
                    for est in ESTIMADORES:
                        p = p_rf if est == "rf" else (p_rf + gb_cache[d]) / 2
                        if escala_alvo == "log":
                            p = np.exp(p)
                        erro.setdefault((nome, enc, est, escala_alvo, semente), {}).update(
                            {ch: abs(p[i] - reais[i]) for i, ch in enumerate(chaves)})
    print(f"  {nome}: concluida", flush=True)
print("grade completa avaliada")

In [ ]:
REGIAO_21 = "ipc_comp"
# erro[(area, enc, est, escala, semente)] = {(dobra, componente, run): erro absoluto}
for nome, df in [(REGIAO_21, datasets[REGIAO_21])]:
    Xs = entradas_por_codificacao(nome, df)
    for enc in ENCODINGS:
        longo = longo_de(df, Xs[enc])
        entradas = [c for c in longo.columns if not c.startswith("_")]
        for escala_alvo in ESCALAS:
            alvo = np.log(longo["_alvo"].to_numpy()) if escala_alvo == "log" else longo["_alvo"].to_numpy()
            gb_cache = {}
            for d in DOBRAS:
                treino = (longo["_dobra"] != d).to_numpy()
                val = (longo["_dobra"] == d).to_numpy()
                if treino.sum() < MIN_TREINO or not val.any():
                    continue
                # patamar e escala do componente, na escala do alvo em uso
                g = pd.Series(alvo[treino]).groupby(longo.loc[treino, "_componente"].to_numpy())
                media, desvio = g.mean(), g.std().fillna(1.0).replace(0, 1.0)
                geral = float(alvo[treino].mean())
                X_tr = matriz(longo, treino, entradas, media, desvio, geral)
                X_val = matriz(longo, val, entradas, media, desvio, geral)
                esc = StandardScaler()
                X_tr, X_val = esc.fit_transform(X_tr), esc.transform(X_val)
                # o Gradient Boosting e deterministico; um ajuste serve a todas as sementes
                gb_cache[d] = GradientBoostingRegressor(n_estimators=300, random_state=0).fit(
                    X_tr, alvo[treino]).predict(X_val)
                chaves = [(d, longo.at[i, "_componente"], int(longo.at[i, "_run"]))
                          for i in np.where(val)[0]]
                reais = longo.loc[val, "_alvo"].to_numpy()
                for semente in SEMENTES_21:
                    p_rf = RandomForestRegressor(n_estimators=300, random_state=semente,
                                                 n_jobs=-1).fit(X_tr, alvo[treino]).predict(X_val)
                    for est in ESTIMADORES:
                        p = p_rf if est == "rf" else (p_rf + gb_cache[d]) / 2
                        if escala_alvo == "log":
                            p = np.exp(p)
                        erro.setdefault((nome, enc, est, escala_alvo, semente), {}).update(
                            {ch: abs(p[i] - reais[i]) for i, ch in enumerate(chaves)})
    print(f"  {nome}: concluida", flush=True)
print("grade completa avaliada")

In [ ]:
esperado = len(datasets) * len(ENCODINGS) * len(ESTIMADORES) * len(ESCALAS) * len(SEMENTES_21)
print(f"grade completa avaliada: {len(erro)} de {esperado} combinacoes")
assert len(erro) == esperado

In [ ]:
# Selecao aninhada: para cada regiao e dobra, a configuracao e escolhida pelo erro
# medio nas DEMAIS dobras. M0 e a entregue, agora sobre as 13 categorias;
# M21 e a formulacao melhorada.
resultado = {}
escolhas = {}
for semente in SEMENTES_21:
    for nome in sorted(datasets.keys()):
        ref = erro[(nome, "13cat", "rf", "linear", semente)]
        dobras_area = sorted({ch[0] for ch in ref})
        for d in dobras_area:
            candidatos = {}
            for enc, est, escala_alvo in product(ENCODINGS, ESTIMADORES, ESCALAS):
                e = erro[(nome, enc, est, escala_alvo, semente)]
                fora = [v for ch, v in e.items() if ch[0] != d]
                candidatos[(enc, est, escala_alvo)] = float(np.mean(fora))
            melhor = min(candidatos, key=candidatos.get)
            escolhas.setdefault(nome, []).append(melhor)
            e_melhor = erro[(nome, melhor[0], melhor[1], melhor[2], semente)]
            for ch, v in ref.items():
                if ch[0] == d:
                    resultado.setdefault(("M0 entregue", semente), {})[(nome,) + ch] = v
                    resultado.setdefault(("M21 melhorada", semente), {})[(nome,) + ch] = e_melhor[ch]

print(f"{'semente':>8}{'M0 entregue':>13}{'M21 melhorada':>15}{'ganho':>8}")
ganhos = []
for semente in SEMENTES_21:
    m0 = np.mean(list(resultado[("M0 entregue", semente)].values()))
    m21 = np.mean(list(resultado[("M21 melhorada", semente)].values()))
    ganhos.append(m0 - m21)
    print(f"{semente:>8}{m0:>13.3f}{m21:>15.3f}{m0 - m21:>+8.3f}")
print(f"\nganho medio {np.mean(ganhos):+.3f} C | positivo em {sum(g > 0 for g in ganhos)} de {len(ganhos)} sementes")

from scipy import stats as _st
chaves_comuns = sorted(set.intersection(*[set(resultado[(m, s)]) for m in ("M0 entregue", "M21 melhorada") for s in SEMENTES_21]))
a = np.mean([[resultado[("M0 entregue", s)][c] for c in chaves_comuns] for s in SEMENTES_21], axis=0)
b = np.mean([[resultado[("M21 melhorada", s)][c] for c in chaves_comuns] for s in SEMENTES_21], axis=0)
dif = a - b
bs = [np.random.default_rng(s).choice(dif, len(dif), replace=True).mean() for s in range(3000)]
print(f"pareado por leitura, media das sementes: ganho {dif.mean():+.3f} C | "
      f"IC95 [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}] | "
      f"wilcoxon p={_st.wilcoxon(a, b).pvalue:.2e}")

print("\nconfiguracao escolhida por regiao (codificacao, estimador, escala), nas dobras e sementes:")
for nome in sorted(datasets.keys()):
    cont = pd.Series([" | ".join(x) for x in escolhas[nome]]).value_counts()
    total = int(cont.sum())
    print(f"  {nome:<15} " + "  ;  ".join(f"{k}: {v}/{total}" for k, v in cont.head(2).items()))

print("\nMAE por regiao:")
print(f"{'regiao':<15}{'entregue':>10}{'melhorada':>11}{'ganho':>8}")
for nome in sorted(datasets.keys()):
    e0 = np.mean([v for s in SEMENTES_21 for c, v in resultado[("M0 entregue", s)].items() if c[0] == nome])
    e1 = np.mean([v for s in SEMENTES_21 for c, v in resultado[("M21 melhorada", s)].items() if c[0] == nome])
    print(f"{nome:<15}{e0:>10.3f}{e1:>11.3f}{e0 - e1:>+8.3f}")

### O que esta seção autoriza afirmar

O ganho reportado é de validação cruzada com a escolha refeita dentro de cada dobra, e portanto já desconta o custo de escolher. A estabilidade da configuração escolhida por região, exibida acima, é o que distingue uma seleção legítima de um ajuste ao ruído: quando a mesma configuração vence em quase todas as dobras e sementes, a escolha reflete estrutura dos dados e não acaso da partição.

O resultado mudou quando a seção foi reexecutada com a versão perfil gerada no colabs_S4. Na primeira medição, a escolha por região rendia 0,037 grau sobre a formulação de referência, com intervalo de confiança entre menos 0,108 e mais 0,187 e ganho em duas das três sementes, o que é indistinguível de ruído. Aquela versão perfil, porém, vinha da terceira sprint e ainda codificava o motivo do ensaio pelos seis indicadores antigos, medidos no tratamento como piores que os seis níveis atuais, além de contar em dobro o sensor duplicado do Oven HMI. As codificações de perfil entravam na grade, portanto, em desvantagem que nada tinha a ver com as perdas. Com as duas versões da base sobre o mesmo motivo, a escolha por região rende 0,229 grau, com intervalo entre 0,123 e 0,330 e ganho nas três sementes.

Duas ressalvas limitam o alcance desse número. A escolha é feita pelo método aninhado aproximado, que a Seção 23 mostrou ser otimista quando os candidatos são muitos, e aqui são doze. E a referência desta seção é a formulação sem o peso próximo ao limite, de modo que o ganho não se compara diretamente com a formulação entregue da Seção 23, que já inclui o peso e escolhe entre as perdas detalhadas e agrupadas; essa comparação, nas mesmas condições, não foi feita.

A formulação melhorada não foi avaliada no conjunto de teste reservado. Essa avaliação, se vier a ser feita, deve ser precedida da decisão explícita de consumir o teste uma segunda vez, registrada na documentação.

## 22. Variabilidade da estimativa: validação cruzada agrupada e repetida

As seções anteriores estimam o erro deixando uma configuração de fora por vez. Esse esquema aproveita ao máximo a base, mas produz uma única partição, e por isso não informa quanto a estimativa flutuaria se as configurações se agrupassem de outro modo.

Esta seção mede essa flutuação. As dezessete configurações de treino são embaralhadas dez vezes e repartidas em cinco dobras a cada vez; nenhuma linha é separada da sua configuração, e o conjunto de teste reservado fica de fora. Dois modelos são avaliados nas mesmas dobras: o entregue e a configuração dominante escolhida por região na Seção 21.

Duas leituras devem ser separadas. O erro absoluto de cada modelo varia com o agrupamento, e a sua dispersão entre repetições é a incerteza da estimativa. O ganho de um modelo sobre o outro, calculado nas mesmas dobras, cancela a dificuldade da dobra, e por isso é mais estável do que os erros absolutos; é essa a razão pela qual o projeto compara sempre de forma pareada.

Uma ressalva de desenho: com dezessete configurações e cinco dobras, cada dobra retém três ou quatro configurações, e como algumas configurações têm dez ensaios e outras um, as dobras saem desbalanceadas. Parte da dispersão medida decorre disso.

A configuração melhorada foi escolhida com os mesmos dados; o seu erro aqui é, portanto, otimista em relação ao aninhado da Seção 21. O que esta seção acrescenta é a dispersão, não o nível.

As células são autocontidas, de modo a poderem ser executadas isoladamente.

In [ ]:

import json
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler

_PART = json.load(open("../colabs_S3/preparacao_testes/particao_treino_teste.json", encoding="utf-8"))
_DOBRA = {int(k): v for k, v in _PART["dobra_por_run_idx"].items()}
_PREF = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
         "perfil_perdas_", "deflector_hole_", "material_coil_support_", "cfg_",
          "reason_", "loss_profile_order_")
_AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling", "mainback", "ipc_comp", "coil_temps"]
# Configuracao dominante por regiao na selecao aninhada da Secao 21, na forma
# (codificacao, estimador, escala do alvo). Transcrita da saida daquela secao;
# se a Secao 21 for reexecutada com outra grade, este dicionario deve segui-la.
# Escolha dominante da Secao 21 reexecutada sobre a versao perfil do colabs_S4,
# que usa o motivo em seis niveis e nao tem o sensor duplicado do Oven HMI. Em
# relacao a transcricao anterior, feita com a versao perfil da terceira sprint,
# mudaram IPC Comp (perfil+ordem -> perfil) e Oven HMI (13cat -> perfil+ordem).
_MELHORADA = {
    "coil_temps": ("13cat", "rf", "log"), "ct_knobs": ("13cat", "rf", "linear"),
    "ipc_comp": ("perfil", "rf", "linear"), "ipc_cooling": ("perfil", "rf", "log"),
    "main_vent_sys": ("13cat", "rf+gb", "linear"), "mainback": ("13cat", "rf", "linear"),
    "oven_hmi": ("perfil+ordem", "rf", "linear"),
}
# Salvaguarda: quando a Secao 21 e executada na mesma sessao, a escolha
# transcrita acima precisa coincidir com a que aquela secao encontrou. Sem esta
# conferencia, uma reexecucao da Secao 21 deixaria o dicionario silenciosamente
# desatualizado, que foi o que ja ocorreu uma vez neste projeto.
if "escolhas" in globals():
    _dominante = {a: Counter(v).most_common(1)[0][0] for a, v in escolhas.items()}
    _divergem = {a: (_MELHORADA[a], _dominante[a]) for a in _dominante
                 if _MELHORADA[a] != _dominante[a]}
    if _divergem:
        print("A transcricao acima diverge da Secao 21 executada nesta sessao.")
        print("A escolha da Secao 21 prevalece; a transcricao deve ser atualizada:")
        for _a, (_t, _r) in sorted(_divergem.items()):
            print(f"    {_a:<15} transcrito {_t}  ->  Secao 21 {_r}")
    else:
        print(f"Transcricao conferida contra a Secao 21: coincide nas {len(_dominante)} regioes.")
    _MELHORADA = _dominante
    _FONTE_ESCOLHA = "Secao 21 desta sessao"
else:
    _FONTE_ESCOLHA = "transcricao no codigo, sem conferencia"
    print(f"Secao 21 nao executada nesta sessao; origem da escolha: {_FONTE_ESCOLHA}.")
print()

_K, _REPETICOES = 5, 10


def _entradas(area, enc):
    # As duas versoes vem desta pasta: o perfil de perdas e gerado na secao 10 do
    # tratamento_datasets_13cat, a partir das treze categorias.
    dp = pd.read_csv(f"dataset_{area}_perfil.csv")
    ordem = [c for c in dp.columns if c.startswith(ORDEM_ACIONAMENTO)]
    if enc == "13cat":
        d13 = pd.read_csv(f"dataset_{area}_13cat.csv")  # pasta desta sprint
        X = d13[[c for c in d13.columns if c.startswith(_PREF) and not c.startswith(ORDEM_ACIONAMENTO)]]
    else:
        X = dp[[c for c in dp.columns if c.startswith(_PREF)]]
        if enc == "perfil":
            X = X.drop(columns=ordem)
    alvos = [c for c in dp.columns if not c.startswith(_PREF) and dp[c].notna().sum() >= 5]
    return X.to_numpy(float), dp[alvos].to_numpy(float)


def _longo(X, Y):
    Xl, yl, rl, cl = [], [], [], []
    for j in range(Y.shape[1]):
        ok = ~np.isnan(Y[:, j])
        Xl.append(X[ok]); yl.append(Y[ok, j]); rl.append(np.arange(len(Y))[ok]); cl.append(np.full(ok.sum(), j))
    Xl, yl, rl, cl = np.vstack(Xl), np.concatenate(yl), np.concatenate(rl), np.concatenate(cl)
    gl = np.array([_DOBRA.get(int(r)) for r in rl], dtype=object)
    ok = np.array([g is not None for g in gl])
    return Xl[ok], yl[ok], cl[ok], gl[ok].astype(int)


def _erro_fora(Xl, yl, cl, tr, va, est, escala, semente):
    yfit = np.log(yl) if escala == "log" else yl
    pat = pd.Series(yfit[tr]).groupby(cl[tr]).mean()
    dsv = pd.Series(yfit[tr]).groupby(cl[tr]).std().fillna(1.0).replace(0, 1.0)
    ger = float(yfit[tr].mean())
    M = lambda m: np.hstack([Xl[m], pd.Series(cl[m]).map(pat).fillna(ger).to_numpy().reshape(-1, 1),
                             pd.Series(cl[m]).map(dsv).fillna(1.0).to_numpy().reshape(-1, 1)])
    s = StandardScaler()
    Xt, Xv = s.fit_transform(M(tr)), s.transform(M(va))
    p = RandomForestRegressor(n_estimators=300, random_state=semente, n_jobs=-1).fit(Xt, yfit[tr]).predict(Xv)
    if est == "rf+gb":
        p = (p + GradientBoostingRegressor(n_estimators=300, random_state=0).fit(Xt, yfit[tr]).predict(Xv)) / 2
    if escala == "log":
        p = np.exp(p)
    return np.abs(p - yl[va])


_dados = {(a, e): _longo(*_entradas(a, e)) for a in _AREAS for e in ("perfil", "perfil+ordem", "13cat")}
_grupos = sorted({g for (_, e), (_, _, _, gl) in _dados.items() if e == "perfil" for g in gl})
print(f"configuracoes de treino: {len(_grupos)} | {_K} dobras | {_REPETICOES} repeticoes\n")

_res = {"entregue": [], "melhorada": []}
_por_area = {m: {a: [] for a in _AREAS} for m in _res}
for rep in range(_REPETICOES):
    perm = np.random.default_rng(100 + rep).permutation(_grupos)
    dobra_de = {g: i % _K for i, g in enumerate(perm)}
    for modelo in _res:
        tudo = []
        for area in _AREAS:
            enc, est, esc = ("13cat", "rf", "linear") if modelo == "entregue" else _MELHORADA[area]
            Xl, yl, cl, gl = _dados[(area, enc)]
            fold = np.array([dobra_de[g] for g in gl])
            e_area = []
            for k in range(_K):
                va, tr = fold == k, fold != k
                if tr.sum() < 30 or not va.any():
                    continue
                e_area.extend(_erro_fora(Xl, yl, cl, tr, va, est, esc, semente=rep))
            _por_area[modelo][area].append(float(np.mean(e_area)))
            tudo.extend(e_area)
        _res[modelo].append(float(np.mean(tudo)))
    print(f"  repeticao {rep}: entregue {_res['entregue'][-1]:.3f} | melhorada {_res['melhorada'][-1]:.3f}", flush=True)


In [ ]:

print("=" * 78)
print(f"VALIDACAO CRUZADA AGRUPADA, {_K} DOBRAS, {_REPETICOES} REPETICOES")
print("=" * 78)
for modelo in _res:
    v = np.array(_res[modelo])
    print(f"{modelo:<10} MAE {v.mean():.3f} C | desvio entre repeticoes {v.std(ddof=1):.3f} | faixa [{v.min():.3f}, {v.max():.3f}]")
_d = np.array(_res["entregue"]) - np.array(_res["melhorada"])
print(f"\nganho da melhorada nas mesmas dobras, por repeticao: {np.round(_d, 3)}")
print(f"ganho medio {_d.mean():+.3f} C | desvio {_d.std(ddof=1):.3f} | positivo em {int((_d > 0).sum())} de {_REPETICOES}")
print("(aninhado, Secao 21: +0,229; deixando uma configuracao de fora, entregue: 6,57)")
print("\npor area, media e desvio entre repeticoes:")
print(f"{'area':<15}{'entregue':>18}{'melhorada':>18}")
for a in _AREAS:
    e, m = np.array(_por_area["entregue"][a]), np.array(_por_area["melhorada"][a])
    print(f"{a:<15}{e.mean():>10.3f} +- {e.std(ddof=1):<5.3f}{m.mean():>10.3f} +- {m.std(ddof=1):<5.3f}")


### Leitura

O erro absoluto de cada modelo flutua, entre repetições, mais do que o ganho de um sobre o outro. Isso não relativiza o ganho: calculado nas mesmas dobras, ele permanece positivo em quase todas as repetições, e é a comparação pareada, e não o nível absoluto, que sustenta a escolha da formulação melhorada.

Com a versão perfil do colabs_S4, a formulação entregue mede 7,087 graus e a melhorada 6,805, com ganho médio de 0,282 grau, positivo em nove das dez repetições. Na primeira execução, com a versão perfil da terceira sprint, o ganho era de menos 0,010 e positivo em cinco das dez, e a mudança tem a mesma origem descrita na leitura da Seção 21. Como a configuração melhorada foi escolhida com os mesmos dados, esse ganho é otimista; o que a seção mede com segurança é a dispersão.

O erro com cinco dobras é sistematicamente maior do que o obtido deixando uma configuração de fora, porque cada modelo treina com cerca de catorze configurações em lugar de dezesseis. O valor obtido com uma configuração de fora situa-se no extremo otimista da faixa; ao reportar o desempenho, convém dar a faixa e não um único número.

A dispersão concentra-se na região das bobinas, onde um sorteio que reúna os ensaios de regime extremo numa mesma dobra deixa o modelo sem ter visto aquele regime, e o erro cresce em dezenas de graus. É a mesma limitação de extrapolação já registrada nas seções anteriores, agora medida como incerteza da própria estimativa.

## 23. Formulação entregue da quarta sprint: estimador e tratamento das perdas por região

As seções anteriores mediram uma grade larga de formulações. Na primeira medição, a seleção entre doze combinações rendeu 0,037 grau, a seleção entre dez rendeu 0,122 e a seleção entre duas, 0,320, o que sugeria que grades estreitas rendem mais. A seleção entre doze foi depois refeita com a versão perfil gerada nesta pasta, que usa o mesmo motivo da base de treze categorias, e passou a render 0,229 grau (Seção 21). As medições indicam que o que decide não é a largura da grade e sim a separação entre os candidatos, pois uma grade de quatro candidatos rendeu 0,184 num caso e custou 0,426 noutro. Onde os candidatos empatam, a seleção vira sorteio.

A grade da Seção 21 não foi medida com o peso próximo ao limite nem pelo método aninhado estrito, e por isso não foi comparada com a formulação desta seção nas mesmas condições. Essa comparação fica registrada como pendente.

Esta seção adota, portanto, uma grade mínima, agora de quatro candidatos por região: dois estimadores, que são a floresta aleatória de trezentas árvores e a média entre floresta e gradiente, por dois tratamentos da coluna de perdas declaradas. O alvo é linear e a codificação é a de treze categorias em todos eles.

Main Vent Sys é a exceção e tem seis candidatos: os quatro anteriores e a média entre Extra Trees e gradiente, com as perdas detalhadas e agrupadas. A mescla foi acrescentada depois da comparação de algoritmos do fim da sprint, em que venceu a floresta com gradiente nessa região em todas as sementes, com e sem o motivo do ensaio e com e sem o peso, por cerca de 0,14 grau. Ela não foi estendida às demais regiões por duas razões medidas: a grade ampla, com vinte candidatos por região e escolha refeita pelo aninhamento estrito, piorou o conjunto em 0,33 grau; e em Coil Temps a mesma mescla reduz o erro médio, mas dobra as aprovações indevidas. Como a mescla foi identificada olhando os resultados, o seu ganho carrega parte do otimismo da escolha, e a diferença está abaixo do piso de ruído da região, de 1,16 grau.

O segundo eixo foi acrescentado por uma auditoria da qualidade do dado bruto. A coluna de perdas chega ao modelo como treze indicadores, cinco deles ligados num único ensaio. Sob validação que remove a configuração inteira, uma coluna ativa num único ensaio nunca está ligada no treino quando aquele ensaio é previsto, de modo que ela não pode ajudar e pode atrapalhar. Os próprios rótulos indicam que boa parte dessa multiplicidade é a mesma coisa escrita de outro modo: o perfil simone aparece com dois nomes e o perfil miro com quatro, sendo que a ordem das bobinas, que distingue três deles, já ocupa coluna própria desde esta sprint. O tratamento agrupado reduz os treze indicadores a quatro níveis mutuamente exclusivos, que são sem perdas, simone, miro e não declarado.

O agrupamento não foi adotado em bloco, e a razão é que medi-lo em bloco não o justifica. Aplicado a todas as regiões, ele rende 0,042 grau, com intervalo de confiança que contém o zero. O que decide é que o efeito é fortemente heterogêneo: três regiões ganham nas três sementes e quatro perdem nas três, e em Mainback a perda é de 0,243 grau com intervalo que exclui o zero. Onde o sinal das perdas é fraco, trocar resolução por suporte ajuda; onde ele é fino, a distinção entre os treze rótulos carrega informação real. Por que Mainback distingue perfis que IPC Comp não distingue permanece sem explicação física, e essa lacuna fica registrada.

Um terceiro elemento foi acrescentado depois, e não é uma escolha por região e sim uma decisão única, aplicada a todas: o ajuste passa a ponderar as leituras pela proximidade ao limite do componente. A razão é que a triagem decide comparando a previsão mais a margem contra o limite, de modo que o erro cometido longe do limite não altera decisão alguma. A forma do peso foi fixada antes de medir e não foi ajustada depois.

Medido com três sementes, o peso deixa o erro médio praticamente inalterado, com ganho de menos 0,011 e intervalo de confiança entre menos 0,063 e mais 0,040, e reduz as aprovações indevidas de 29,7 para 23,3, com a detecção subindo de 84,6 para 87,9 por cento. A redução ocorre nas três sementes. O custo são quatro reprovações desnecessárias a mais, que correspondem a reensaio, e é a troca que o projeto já aceitou ao escolher a margem de oito graus.

A escolha entre candidatos é aninhada. Para cada região e cada dobra, o candidato é escolhido pelo erro médio nas demais dobras, e só então aplicado à dobra retida. Sem isso, o número mediria a própria escolha em vez da capacidade de generalizar.

In [ ]:
import json
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from scipy import stats
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler

_PART = json.load(open("../colabs_S3/preparacao_testes/particao_treino_teste.json", encoding="utf-8"))
_DOBRA = {int(k): v for k, v in _PART["dobra_por_run_idx"].items()}
_TESTE = set(_PART["run_idx_teste"])
_LIM = pd.read_csv("limites_ativos.csv").set_index("tc_name")["limite_num"]
_PREF = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
         "deflector_hole_", "material_coil_support_", "reason_", "loss_profile_order_")
_ORDEM = "loss_profile_order_"
_PERDAS = "losses_"
_AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling", "mainback", "ipc_comp", "coil_temps"]
_SEMENTES = (42, 1, 2)
_ESTIMADORES = ("rf", "rf+gb")
_TRATAMENTOS = ("detalhado", "agrupado")
_CANDIDATOS = [(e, t) for e in _ESTIMADORES for t in _TRATAMENTOS]
# Main Vent Sys recebe dois candidatos a mais: a media entre Extra Trees e gradiente,
# com as perdas detalhadas e agrupadas. A mescla foi identificada depois da comparacao
# de algoritmos da quarta sprint, em que venceu a entrega nessa regiao em todas as
# sementes e variantes; foi acrescentada so aqui, e nao em todas as regioes, porque a
# grade ampla medida com aninhamento estrito piorou o conjunto quando todas as regioes
# escolhiam entre vinte candidatos, e porque em Coil Temps a mesma mescla reduz o erro
# medio mas dobra as aprovacoes indevidas.
_EXTRA_POR_AREA = {"main_vent_sys": [("et+gb", "detalhado"), ("et+gb", "agrupado")]}
_CANDIDATOS_DE = {a: _CANDIDATOS + _EXTRA_POR_AREA.get(a, []) for a in _AREAS}
_MARGEM = 8
_PESO = (317 - 44) / (74 - 25)   # razao de custo revelada pela escolha da margem


def _familia_de_perdas(coluna):
    """O perfil dissipado, sem a ordem das bobinas e sem o tipo de ensaio.

    A ordem das bobinas ja e coluna propria e o tipo de ensaio ja e a linha de
    run case, de modo que o que resta de informacao nova nesta coluna e qual
    perfil de potencia foi imposto.
    """
    t = coluna[len(_PERDAS):].lower()
    if "no losses" in t:
        return "none"
    if "simone" in t:
        return "simone"
    if "miro" in t:
        return "miro"
    return "other"


def _bloco_agrupado(d, colunas_perdas):
    """Niveis mutuamente exclusivos no lugar dos indicadores de perdas.

    O nivel de nao declarado e derivado da ausencia de qualquer indicador, e nao
    de um rotulo proprio, porque a planilha deixa a celula vazia nesses ensaios.
    """
    fam = np.array([_familia_de_perdas(c) for c in colunas_perdas])
    A = d[colunas_perdas].to_numpy(float)
    niveis = sorted(set(fam.tolist()) | {"not_declared"})
    M = np.zeros((len(d), len(niveis)))
    for j, nivel in enumerate(niveis):
        if nivel == "not_declared":
            M[:, j] = (A.sum(axis=1) == 0).astype(float)
        else:
            M[:, j] = (A[:, fam == nivel].sum(axis=1) > 0).astype(float)
    assert (M.sum(axis=1) == 1).all(), "os niveis de perdas deixaram de ser exclusivos"
    return M, niveis


def _entradas(area, tratamento):
    """Matriz de entrada da regiao, sem as colunas de ordem de chaveamento, que
    foram medidas como prejudiciais."""
    d = pd.read_csv(f"dataset_{area}_13cat.csv")
    cols = [c for c in d.columns if c.startswith(_PREF) and not c.startswith(_ORDEM)]
    perdas = [c for c in cols if c.startswith(_PERDAS)]
    assert perdas, f"{area}: bloco de perdas ausente"
    if tratamento == "detalhado":
        X = d[cols].to_numpy(float)
    else:
        outras = [c for c in cols if c not in perdas]
        M, _ = _bloco_agrupado(d, perdas)
        X = np.hstack([d[outras].to_numpy(float), M])
    alvos = [c for c in d.columns if not c.startswith(_PREF) and d[c].notna().sum() >= 5]
    # Toda coluna tratada como alvo precisa ser um termopar da tabela de limites. Sem
    # esta verificacao, uma coluna de configuracao cujo prefixo nao estivesse em _PREF
    # seria classificada como alvo e prevista como se fosse temperatura, sem erro
    # algum. Foi o que aconteceu com o motivo do ensaio quando mudou de nome de cfg_
    # para reason_, primeiro neste caderno e depois no de busca aleatoria.
    _estranhas = [c for c in alvos if c not in _LIM.index]
    assert not _estranhas, (f"{area}: colunas tratadas como alvo que nao sao termopares: "
                            f"{_estranhas}. Declare o prefixo em _PREF.")
    return d, X, alvos


def _longo(d, X, alvos):
    Y = d[alvos].to_numpy(float)
    Xl, yl, rl, cl = [], [], [], []
    for j in range(Y.shape[1]):
        ok = ~np.isnan(Y[:, j])
        Xl.append(X[ok]); yl.append(Y[ok, j])
        rl.append(np.arange(len(d))[ok]); cl.append(np.full(ok.sum(), j))
    Xl, yl = np.vstack(Xl), np.concatenate(yl)
    rl, cl = np.concatenate(rl), np.concatenate(cl)
    return Xl, yl, cl, rl, [alvos[j] for j in cl]


def _carrega(area, tratamento):
    """Formato longo da regiao, sem o conjunto de teste reservado."""
    d, X, alvos = _entradas(area, tratamento)
    Xl, yl, cl, rl, comps = _longo(d, X, alvos)
    gl = np.array([_DOBRA.get(int(r)) for r in rl], dtype=object)
    ok = np.array([g is not None for g in gl]) & ~np.isin(rl, list(_TESTE))
    return (Xl[ok], yl[ok], cl[ok], gl[ok].astype(int), rl[ok],
            [c for c, m in zip(comps, ok) if m])


def _peso_perto_do_limite(y, comps):
    """Peso maior para as leituras proximas do limite do componente.

    A triagem decide comparando a previsao mais a margem contra o limite, de modo
    que o erro cometido longe do limite nao muda decisao alguma. Ponderar o ajuste
    pela proximidade ao limite concentra a capacidade do modelo onde a decisao
    acontece.

    A forma foi fixada antes de medir e nao foi ajustada depois: peso um mais dois
    vezes uma gaussiana de largura quinze graus, de modo que uma leitura sobre o
    limite pesa tres vezes e uma a trinta graus dele pesa praticamente um.
    Componentes sem limite tabelado recebem peso um.
    """
    lim = np.array([_LIM.get(c, np.nan) for c in comps], float)
    d = np.abs(y - lim)
    return np.where(np.isnan(lim), 1.0, 1.0 + 2.0 * np.exp(-(np.nan_to_num(d, nan=1e9) / 15.0) ** 2))


def _ajusta(Xt, yt, Xv, estimador, semente, peso=None):
    """rf: floresta; rf+gb e et+gb: media da floresta ou do Extra Trees com o gradiente."""
    base = ExtraTreesRegressor if estimador.startswith("et") else RandomForestRegressor
    p = base(n_estimators=300, random_state=semente, n_jobs=-1).fit(
        Xt, yt, sample_weight=peso).predict(Xv)
    if estimador.endswith("+gb"):
        p = (p + GradientBoostingRegressor(n_estimators=300, random_state=semente).fit(
            Xt, yt, sample_weight=peso).predict(Xv)) / 2
    return p


def _fora_da_dobra(area, candidato, semente):
    """Erro de cada leitura quando a sua configuracao esta fora do treino."""
    estimador, tratamento = candidato
    Xl, yl, cl, gl, rl, comps = _carrega(area, tratamento)
    saida = {}
    for k in sorted(set(gl)):
        va, tr = gl == k, gl != k
        if tr.sum() < 30 or not va.any():
            continue
        pat = pd.Series(yl[tr]).groupby(cl[tr]).mean()
        dsv = pd.Series(yl[tr]).groupby(cl[tr]).std().fillna(1.0).replace(0, 1.0)
        ger = float(yl[tr].mean())
        M = lambda m: np.hstack([Xl[m],
                                 pd.Series(cl[m]).map(pat).fillna(ger).to_numpy().reshape(-1, 1),
                                 pd.Series(cl[m]).map(dsv).fillna(1.0).to_numpy().reshape(-1, 1)])
        s = StandardScaler()
        Xt, Xv = s.fit_transform(M(tr)), s.transform(M(va))
        p = _ajusta(Xt, yl[tr], Xv, estimador, semente,
                    _peso_perto_do_limite(yl[tr], [comps[i] for i in np.where(tr)[0]]))
        for idx, i in enumerate(np.where(va)[0]):
            saida[(int(rl[i]), comps[i])] = (p[idx] - yl[i], yl[i], comps[i])
    return saida


_ndet = len(_entradas("mainback", "detalhado")[1][0])
_nagr = len(_entradas("mainback", "agrupado")[1][0])
_d0 = pd.read_csv("dataset_mainback_13cat.csv")
_, _niveis = _bloco_agrupado(_d0, [c for c in _d0.columns if c.startswith(_PERDAS)])
print(f"Niveis do tratamento agrupado: {_niveis}")
print(f"Colunas de entrada: {_ndet} no tratamento detalhado, {_nagr} no agrupado")
print()

_erro = {}
for _s in _SEMENTES:
    for _a in _AREAS:
        for _c in _CANDIDATOS_DE[_a]:
            _erro.setdefault((_c, _s), {})[_a] = _fora_da_dobra(_a, _c, _s)
    print(f"  semente {_s} concluida", flush=True)

In [ ]:
# A formulacao anterior, usada como referencia do ganho, e a escolha de estimador
# por regiao com as perdas detalhadas, que foi a entrega antes desta auditoria.
_ANTERIOR = {"oven_hmi": "rf", "ct_knobs": "rf", "main_vent_sys": "rf+gb",
             "ipc_cooling": "rf", "mainback": "rf", "ipc_comp": "rf+gb", "coil_temps": "rf+gb"}

_res = {n: {} for n in ("anterior", "escolhido")}
_escolhas = {a: [] for a in _AREAS}
for _s in _SEMENTES:
    for area in _AREAS:
        ref = _erro[((_ANTERIOR[area], "detalhado"), _s)][area]
        for d in sorted({_DOBRA[ch[0]] for ch in ref}):
            cand = {c: float(np.mean([abs(v[0]) for ch, v in _erro[(c, _s)][area].items()
                                      if _DOBRA[ch[0]] != d])) for c in _CANDIDATOS_DE[area]}
            melhor = min(cand, key=cand.get)
            _escolhas[area].append(melhor)
            for ch in ref:
                if _DOBRA[ch[0]] == d:
                    _res["anterior"].setdefault(_s, {})[(area,) + ch] = ref[ch]
                    _res["escolhido"].setdefault(_s, {})[(area,) + ch] = _erro[(melhor, _s)][area][ch]

print("Candidato escolhido por regiao, nas dobras e sementes:\n")
_DOMINANTE = {}
for area in _AREAS:
    c = Counter(_escolhas[area])
    _DOMINANTE[area] = c.most_common(1)[0][0]
    print(f"  {area:<15}" + "  ;  ".join(f"{e}|{t}: {n}/{len(_escolhas[area])}"
                                         for (e, t), n in c.most_common()))
print("\nRegra dominante, que e a que a entrega adota:")
for a, (e, t) in _DOMINANTE.items():
    print(f"  {a:<15} {e:<6} perdas {t}")

print("\nErro medio absoluto de cada candidato, por regiao, para leitura do porque:\n")
_chaves = {a: sorted(_erro[(_CANDIDATOS[0], _SEMENTES[0])][a]) for a in _AREAS}
_TODOS = _CANDIDATOS + [c for a in _AREAS for c in _EXTRA_POR_AREA.get(a, []) if c not in _CANDIDATOS]
print(f"{'regiao':<16}" + "".join(f"{e + ' ' + t:>22}" for e, t in _TODOS))
for area in _AREAS:
    linha = []
    for c in _TODOS:
        if c not in _CANDIDATOS_DE[area]:
            linha.append("-")
            continue
        v = np.mean([np.mean([abs(_erro[(c, s)][area][ch][0]) for ch in _chaves[area]])
                     for s in _SEMENTES])
        linha.append(f"{v:.3f}")
    print(f"{area:<16}" + "".join(f"{x:>22}" for x in linha))


def _metricas(mapa):
    dif = np.array([v[0] for v in mapa.values()], float)
    real = np.array([v[1] for v in mapa.values()], float)
    comps = [v[2] for v in mapa.values()]
    fa = fr = det = tot = 0
    for r, d, c in zip(real, dif, comps):
        lim = _LIM.get(c)
        if lim is None or pd.isna(lim):
            continue
        estoura, alarme = r >= lim, (r + d) + _MARGEM >= lim
        tot += estoura; fa += estoura and not alarme
        fr += (not estoura) and alarme; det += estoura and alarme
    return {"n": len(dif), "MAE": float(np.mean(np.abs(dif))),
            "RMSE": float(np.sqrt(np.mean(dif ** 2))),
            "R2": float(1 - (dif ** 2).sum() / ((real - real.mean()) ** 2).sum()),
            "vies": float(dif.mean()), "MAPE_%": float(np.mean(np.abs(dif) / real) * 100),
            "dentro_10_%": float(np.mean(np.abs(dif) / real < 0.10) * 100),
            "custo": float(np.mean(np.where(dif < 0, -dif * _PESO, dif))),
            "falsos_aprovados": fa, "falsos_reprovados": fr,
            "deteccao_%": (100.0 * det / tot if tot else float("nan"))}


_comuns = sorted(set.intersection(*[set(_res["anterior"][s]) for s in _SEMENTES]))
_a = np.mean([[abs(_res["anterior"][s][c][0]) for c in _comuns] for s in _SEMENTES], axis=0)
_b = np.mean([[abs(_res["escolhido"][s][c][0]) for c in _comuns] for s in _SEMENTES], axis=0)
_d = _a - _b
_bs = [np.random.default_rng(i).choice(_d, len(_d), replace=True).mean() for i in range(3000)]
print(f"\nMAE da formulacao anterior, estimador por regiao e perdas detalhadas: {_a.mean():.3f} C")
print(f"MAE da formulacao com estimador e tratamento das perdas por regiao:   {_b.mean():.3f} C")
print(f"ganho {_d.mean():+.3f} C | IC95 [{np.percentile(_bs, 2.5):+.3f}, {np.percentile(_bs, 97.5):+.3f}] "
      f"| wilcoxon p={stats.wilcoxon(_a, _b).pvalue:.2e}")

print("\nMetricas por regiao da formulacao adotada:\n")
_linhas = {}
for area in _AREAS + ["AGREGADO"]:
    por_s = []
    for s in _SEMENTES:
        m = {k: v for k, v in _res["escolhido"][s].items() if area == "AGREGADO" or k[0] == area}
        por_s.append(_metricas(m))
    _linhas[area] = {k: float(np.mean([p[k] for p in por_s])) for k in por_s[0]}
_T = pd.DataFrame(_linhas).T
print(_T.round(3).to_string())

### Leitura

A escolha por região rende 0,146 grau sobre a formulação anterior, que já escolhia o estimador por região com as perdas detalhadas, com intervalo de confiança entre 0,062 e 0,233 grau, que não contém o zero. O teste de Wilcoxon não acompanha o intervalo com a mesma força, com p de 0,077, acima do nível usual de cinco por cento, de modo que o ganho é consistente na média, mas não se distribui uniformemente entre as leituras.

A escolha é unânime em três regiões, Oven HMI, Main Vent Sys e Mainback, com cinquenta e uma seleções em cinquenta e uma, e quase unânime em outras três, com quarenta e sete a quarenta e nove. A exceção é CT Knobs, em que a floresta com perdas detalhadas vence em trinta e cinco escolhas e a floresta com perdas agrupadas em treze: ali os candidatos quase empatam, a 2,952 e 2,957 graus, e a escolha oscila sem custo relevante. É a condição sob a qual a lição das seções 21 e 22 diz que a seleção se sustenta: o que decide não é a largura da grade e sim a separação entre os candidatos.

O movimento de maior peso está em IPC Comp, que é também a região de maior massa de erro redutível. Ali a seleção agrupa as perdas e troca o estimador, da média entre floresta e gradiente para a floresta pura, e a região passa de 7,027 para 6,677 graus, numa região que responde por metade das leituras do projeto. O segundo movimento está em Main Vent Sys, em que a média entre Extra Trees e gradiente é escolhida em todas as dobras e sementes e a região passa de 3,890 para 3,739 graus.

Quanto à triagem de segurança, a formulação adotada registra 23,3 aprovações indevidas, 157 reprovações desnecessárias e detecção de 87,9 por cento. A troca em Main Vent Sys não altera esses números, porque a região não tem leituras acima do limite.

Registre-se, por fim, que esta seção não toca o conjunto de teste reservado. A escolha da regra foi feita inteiramente sobre a validação cruzada.


### Avaliação no conjunto de teste reservado

A regra dominante da seleção é aplicada uma única vez aos dez ensaios reservados, com os sete modelos ajustados sobre a totalidade dos quarenta e seis ensaios de desenvolvimento, que é a condição do modelo efetivamente entregue.

Uma ressalva precede o número e não deve ser omitida de nenhum relato que o cite. Este conjunto já foi lido diversas vezes ao longo do projeto, e cada leitura consome parte da proteção que ele oferece. Além disso, a sua composição é desequilibrada: o tipo de ensaio broil c1 não tem representação alguma nele, e o tipo pyro, que é o regime mais quente e o que mais produz excessos de limite, comparece com um único ensaio entre os oito existentes.

Por essas duas razões, a documentação do projeto registra desde a terceira sprint que a leitura conservadora do desempenho é a da validação cruzada, que cobre os dois regimes. Essa orientação foi escrita quando o conjunto reservado favorecia o modelo, e é mantida agora.

In [ ]:
# A regra e escrita como constante, e nao lida de _DOMINANTE, para que a entrega
# nao dependa da ordem de execucao das celulas. A celula seguinte confere que as
# duas coincidem e relata divergencia em vez de aborta-la.
_REGRA = {"oven_hmi": ("rf", "agrupado"), "ct_knobs": ("rf", "detalhado"),
          "main_vent_sys": ("et+gb", "detalhado"), "ipc_cooling": ("rf", "agrupado"),
          "mainback": ("rf", "detalhado"), "ipc_comp": ("rf", "agrupado"),
          "coil_temps": ("rf+gb", "detalhado")}
_SEMENTE_ENTREGA = 42

if "_DOMINANTE" in dir():
    _divergem = [a for a in _AREAS if _DOMINANTE[a] != _REGRA[a]]
    if _divergem:
        print("Aviso: a selecao aninhada desta execucao diverge da regra escrita em:")
        for a in _divergem:
            print(f"  {a:<15} selecionado {_DOMINANTE[a]} | escrito {_REGRA[a]}")
        print()
    else:
        print("A regra escrita coincide com a selecao aninhada desta execucao.\n")


def _completo(area, tratamento):
    """Formato longo conservando os ensaios reservados, marcados a parte."""
    d, X, alvos = _entradas(area, tratamento)
    return _longo(d, X, alvos)


print(f"Ensaios de desenvolvimento: {56 - len(_TESTE)} | reservados: {len(_TESTE)}")
print("Regra aplicada:")
for a, (e, t) in _REGRA.items():
    print(f"  {a:<15} {e:<6} perdas {t}")
print()

_linhas_teste = []
for area in _AREAS:
    _est, _trat = _REGRA[area]
    Xl, yl, cl, rl, comps = _completo(area, _trat)
    tes = np.isin(rl, list(_TESTE))
    tr = ~tes
    # Patamar e escala vem apenas do desenvolvimento, como em toda a modelagem.
    pat = pd.Series(yl[tr]).groupby(cl[tr]).mean()
    dsv = pd.Series(yl[tr]).groupby(cl[tr]).std().fillna(1.0).replace(0, 1.0)
    ger = float(yl[tr].mean())
    M = lambda m: np.hstack([Xl[m],
                             pd.Series(cl[m]).map(pat).fillna(ger).to_numpy().reshape(-1, 1),
                             pd.Series(cl[m]).map(dsv).fillna(1.0).to_numpy().reshape(-1, 1)])
    s = StandardScaler()
    Xt, Xv = s.fit_transform(M(tr)), s.transform(M(tes))
    p = _ajusta(Xt, yl[tr], Xv, _est, _SEMENTE_ENTREGA,
                _peso_perto_do_limite(yl[tr], [comps[i] for i in np.where(tr)[0]]))
    for i, idx in enumerate(np.where(tes)[0]):
        _linhas_teste.append({"area": area, "componente": comps[idx], "real": yl[idx],
                              "pred": p[i]})

_T = pd.DataFrame(_linhas_teste)
_T["dif"] = _T["pred"] - _T["real"]
_T["limite"] = _T["componente"].map(_LIM)

print("Desempenho no conjunto de teste reservado, por area:\n")
_saida = {}
for area in _AREAS + ["AGREGADO"]:
    g = _T if area == "AGREGADO" else _T[_T["area"] == area]
    d, r = g["dif"].to_numpy(), g["real"].to_numpy()
    _saida[area] = {"n": len(g), "MAE": np.abs(d).mean(), "RMSE": float(np.sqrt((d ** 2).mean())),
                    "R2": float(1 - (d ** 2).sum() / ((r - r.mean()) ** 2).sum()),
                    "vies": d.mean(), "MAPE_%": float((np.abs(d) / r).mean() * 100),
                    "dentro_10_%": float((np.abs(d) / r < 0.10).mean() * 100)}
print(pd.DataFrame(_saida).T.round(3).to_string())

_L = _T.dropna(subset=["limite"]).copy()
_L["estoura"] = _L["real"] >= _L["limite"]
print(f"\nLeituras com limite: {len(_L)} | que atingem o limite: {int(_L['estoura'].sum())}\n")
print(f"{'margem':>7}{'falsos aprovados':>19}{'falsos reprovados':>19}{'deteccao':>11}")
for m in (0, 5, 8, 10, 12):
    alarme = _L["pred"] + m >= _L["limite"]
    fa = int((_L["estoura"] & ~alarme).sum()); fr = int((~_L["estoura"] & alarme).sum())
    det = 100 * (_L["estoura"] & alarme).sum() / _L["estoura"].sum()
    print(f"{m:>7}{fa:>19}{fr:>19}{det:>10.1f}%")

print("\nReferencias documentadas, no mesmo conjunto:")
print("  Develop, perfil e motivo em seis indicadores, floresta:   MAE 4,37 | dentro de 10%: 85,7%")
print("  S4 antes da escolha de estimador, floresta em todas:      MAE 5,15 | dentro de 10%: 83,9%")
print("  S4 com escolha de estimador e perdas detalhadas:          MAE 5,17 | dentro de 10%: 82,0%")

### Leitura do conjunto reservado

A formulação entregue, já com o peso próximo ao limite e com a média entre Extra Trees e gradiente em Main Vent Sys, mede 5,154 graus no conjunto reservado, contra 5,17 da formulação que escolhia apenas o estimador e 5,15 da formulação de referência. A divergência com a versão anterior do projeto, que mede 4,37 ali, permanece, e diminuiu pouco, de 0,80 para 0,78 grau. A fração de leituras dentro de dez por cento passou de 82,0 para 82,6 por cento, ainda abaixo dos 83,9 da formulação de referência. Na triagem com margem de oito graus, das 63 leituras que atingem o limite, 12 passam sem alarme, com 47 alarmes desnecessários e detecção de 81,0 por cento.

No conjunto reservado, a diferença para as duas formulações anteriores é de dois a três centésimos de grau, o que equivale a um empate. O resultado não confirma o ganho medido na validação cruzada, mas também não o contradiz: ao contrário das formulações testadas por busca de hiperparâmetros, que venciam na validação e perdiam no reservado, sinal de ajuste às dobras, esta não perde no erro médio em nenhum dos dois. Cabe registrar que duas mudanças que melhoraram a validação cruzada pioraram o conjunto reservado. O peso próximo ao limite levou o erro de 4,896 para 5,143 graus. A troca em Main Vent Sys, que reduz o erro da região de 3,890 para 3,739 graus na validação, eleva o erro da região no conjunto reservado de 3,538 para 3,708, com cinquenta leituras. A escolha de manter as duas seguiu a leitura conservadora descrita abaixo.

Ainda assim, o confronto entre as duas leituras não está resolvido, e a recomendação do projeto continua a ser a de tratar a validação cruzada como leitura conservadora, por cobrir os dois regimes de ensaio.

O caminho para resolver a divergência não é estatístico e sim experimental: são necessários ensaios de broil c1, hoje ausentes do conjunto reservado, e mais ensaios de pyro, hoje representados por um único caso. Enquanto eles não existirem, qualquer conclusão sobre qual das duas leituras descreve melhor o uso real permanecerá sem sustentação.

## 24. Diagnóstico do coeficiente de determinação do Oven HMI

O Oven HMI tem o menor erro absoluto das sete regiões e, ao mesmo tempo, um coeficiente de determinação próximo de zero, o que chamou a atenção do grupo. Esta seção investiga a causa e testa abordagens alternativas: a presença ou ausência das colunas do motivo do ensaio, a mescla da floresta com a regressão linear e a mescla de floresta, gradiente e rede neural, esta última usada por outro grupo da turma.

O protocolo é o do projeto: validação por configuração em dezessete dobras, patamar e escala do componente calculados apenas no treino de cada dobra, peso próximo ao limite e três sementes. As mesclas são médias simples das previsões. A célula é autocontida, lê apenas arquivos do colabs_S4 e não consulta o conjunto de teste reservado.

O coeficiente de determinação é reportado de três formas. O conjunto é o usado até aqui, calculado sobre a variação total das 135 leituras. O por componente é a média dos coeficientes calculados dentro de cada componente. O dentro compara o erro com a variação de cada leitura em torno da média do seu componente, e mede o quanto o modelo explica além de saber qual componente está sendo previsto.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from joblib import Parallel, delayed
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

_SEM = (42, 1, 2)
_part = pd.read_csv("particao_treino_teste.csv")
_DOB = dict(zip(_part["run_idx"], _part["dobra"]))
_RES = set(_part.loc[_part["conjunto"] == "teste", "run_idx"])
_LIMO = pd.read_csv("limites_ativos.csv").set_index("tc_name")["limite_num"]
_d = pd.read_csv("dataset_oven_hmi_13cat.csv")
_d["run_idx"] = np.arange(len(_d))
_BASE = ("blower_direction_", "housing_", "board_position_", "run_case_", "deflector_hole_",
         "material_coil_support_")
_PERD = [c for c in _d.columns if c.startswith("losses_")]
_MOT = [c for c in _d.columns if c.startswith("reason_")]
_ALV = [c for c in _d.columns if not c.startswith(_BASE + ("losses_", "reason_", "loss_profile_order_"))
        and c != "run_idx"]


def _familia(c):
    t = c[len("losses_"):].lower()
    return next((n for k, n in (("no losses", "none"), ("simone", "simone"), ("miro", "miro")) if k in t), "other")


def _longo(trat, com_motivo):
    if trat == "detalhado":
        perd = _d[_PERD]
    else:
        fam = np.array([_familia(c) for c in _PERD]); A = _d[_PERD].to_numpy(float)
        perd = pd.DataFrame({f"perdas_{n}": (A[:, fam == n].sum(axis=1) > 0).astype(float) for n in sorted(set(fam))})
        perd["perdas_nao_declarado"] = (A.sum(axis=1) == 0).astype(float)
    X = pd.concat([_d[[c for c in _d.columns if c.startswith(_BASE)]], perd] + ([_d[_MOT]] if com_motivo else []), axis=1)
    X["run_idx"] = _d["run_idx"]
    L = X.join(_d[_ALV]).melt(id_vars=list(X.columns), value_vars=_ALV, var_name="componente", value_name="y")
    L["dobra"] = L["run_idx"].map(_DOB)
    L = L[L["dobra"].notna() & L["y"].notna() & ~L["run_idx"].isin(_RES)].reset_index(drop=True)
    return L, [c for c in X.columns if c != "run_idx"]


def _peso(y, comps):
    lim = np.array([_LIMO.get(c, np.nan) for c in comps], float)
    return np.where(np.isnan(lim), 1.0, 1.0 + 2.0 * np.exp(-(np.nan_to_num(np.abs(y - lim), nan=1e9) / 15.0) ** 2))


_MODELOS = {
    "floresta (RF)": lambda s: RandomForestRegressor(n_estimators=300, random_state=s, n_jobs=1),
    "gradiente (GB)": lambda s: GradientBoostingRegressor(n_estimators=300, random_state=s),
    "regressao linear": lambda s: make_pipeline(StandardScaler(), LinearRegression()),
    "ridge": lambda s: make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    "rede neural (MLP)": lambda s: TransformedTargetRegressor(regressor=make_pipeline(
        StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1.0, max_iter=3000, random_state=s)),
        transformer=StandardScaler()),
}
_MESCLAS = {"RF + regressao linear": ("floresta (RF)", "regressao linear"),
            "RF + ridge": ("floresta (RF)", "ridge"), "RF + GB": ("floresta (RF)", "gradiente (GB)"),
            "RF + MLP": ("floresta (RF)", "rede neural (MLP)"),
            "RF + GB + MLP": ("floresta (RF)", "gradiente (GB)", "rede neural (MLP)")}


def _ajusta(m, A, y, w):
    if isinstance(m, TransformedTargetRegressor):
        return m.fit(A, y, mlpregressor__sample_weight=w)
    if hasattr(m, "steps"):
        return m.fit(A, y, **{m.steps[-1][0] + "__sample_weight": w})
    return m.fit(A, y, sample_weight=w)


def _dobra(L, feat, k, s):
    tr, va = L[L["dobra"] != k], L[L["dobra"] == k]
    st = tr.groupby("componente")["y"].agg(patamar="mean", escala="std")
    A, B = tr.join(st, on="componente"), va.join(st, on="componente")
    F = feat + ["patamar", "escala"]
    w = _peso(A["y"].to_numpy(), A["componente"].to_numpy())
    out = {n: _ajusta(f(s), A[F].to_numpy(float), A["y"].to_numpy(), w).predict(B[F].to_numpy(float))
           for n, f in _MODELOS.items()}
    for n, ps in _MESCLAS.items():
        out[n] = np.mean([out[p] for p in ps], axis=0)
    out["media historica do componente"] = B["patamar"].to_numpy()
    return va.index.to_numpy(), out


def _mede(y, p, comp):
    e = p - y
    mc = pd.Series(y).groupby(comp).transform("mean").to_numpy()
    r2c = [1 - (e[comp == c] ** 2).sum() / ((y[comp == c] - y[comp == c].mean()) ** 2).sum() for c in np.unique(comp)]
    return {"MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean()),
            "R2 conjunto": 1 - (e ** 2).sum() / ((y - y.mean()) ** 2).sum(), "R2 por componente": np.mean(r2c),
            "R2 dentro": 1 - (e ** 2).sum() / ((y - mc) ** 2).sum(), "dentro de 10%": 100 * np.mean(np.abs(e) / y < 0.10)}


_t0 = time.time()
_linhas, _PRED = [], {}
for _trat, _mot in (("agrupado", True), ("agrupado", False), ("detalhado", True), ("detalhado", False)):
    _L, _feat = _longo(_trat, _mot)
    _y, _c = _L["y"].to_numpy(), _L["componente"].to_numpy()
    _por = []
    for _s in _SEM:
        _res = Parallel(n_jobs=-1)(delayed(_dobra)(_L, _feat, k, _s) for k in sorted(set(_L["dobra"])))
        _P = {n: np.full(len(_L), np.nan) for n in _res[0][1]}
        for _idx, _out in _res:
            for n, p in _out.items():
                _P[n][_idx] = p
        _por.append({n: _mede(_y, p, _c) for n, p in _P.items()})
        if (_trat, _mot) == ("agrupado", True):
            _PRED[_s] = (_L, _P["floresta (RF)"])
    for n in _por[0]:
        _linhas.append({"perdas": _trat, "motivo": "com" if _mot else "sem", "modelo": n,
                        **pd.DataFrame([p[n] for p in _por]).mean().to_dict()})
TAB_OVEN = pd.DataFrame(_linhas)
print(f"{len(_linhas)} combinacoes em {(time.time() - _t0) / 60:.1f} min, {len(_y)} leituras\n")
pd.set_option("display.width", 200)
for (_t, _m), _g in TAB_OVEN.groupby(["perdas", "motivo"], sort=False):
    print(f"--- perdas {_t}, motivo {_m} (media de 3 sementes)")
    print(_g.drop(columns=["perdas", "motivo"]).sort_values("MAE").round(3).to_string(index=False), "\n")

### De onde vem o coeficiente baixo

A célula seguinte localiza os erros da floresta na formulação entregue, com perdas agrupadas e motivo, e compara os ensaios de tipo pyro, que compartilham a mesma configuração declarada.

In [ ]:
_L0 = _PRED[42][0].copy()
_L0["pred"] = np.mean([_PRED[s][1] for s in _SEM], axis=0)
_L0["erro"] = _L0["pred"] - _L0["y"]
_rc = [c for c in _d.columns if c.startswith("run_case_")]
_L0["tipo"] = _L0["run_idx"].map(lambda r: next((c[9:] for c in _rc if _d.loc[r, c] == 1), "?"))
_L0["motivo"] = _L0["run_idx"].map(lambda r: next((c[7:] for c in _MOT if _d.loc[r, c] == 1), "?"))
_q = _L0["erro"] ** 2
_top = _q.sort_values(ascending=False)
print("Maiores erros absolutos:")
print(_L0.reindex(_L0["erro"].abs().sort_values(ascending=False).index)
      [["run_idx", "componente", "tipo", "motivo", "y", "pred", "erro"]].head(8).round(2).to_string(index=False))
print(f"\nParcela do erro quadratico: 5 maiores leituras {100 * _top.head(5).sum() / _q.sum():.0f}% | "
      f"10 maiores {100 * _top.head(10).sum() / _q.sum():.0f}%")
print("\nErro por tipo de ensaio:")
print(_L0.groupby("tipo").agg(leituras=("y", "size"), MAE=("erro", lambda e: e.abs().mean()),
                              RMSE=("erro", lambda e: np.sqrt((e ** 2).mean()))).round(2).to_string())

_pyro = [r for r in range(len(_d)) if _d.loc[r, "run_case_pyro"] == 1 and r not in _RES]
_cfg = [c for c in _d.columns if c.startswith(_BASE + ("losses_",))]
print("\nEnsaios pyro de desenvolvimento, dobra, configuracao declarada e temperatura media:")
_rotulos = {}
_tab = pd.DataFrame({"ensaio": _pyro, "dobra": [int(_DOB[r]) for r in _pyro],
                     "configuracao": [_rotulos.setdefault(tuple(_d.loc[r, _cfg]), "ABCDEFGH"[len(_rotulos)]) for r in _pyro],
                     "motivo": [_L0.loc[_L0.run_idx == r, "motivo"].iloc[0] for r in _pyro],
                     "temperatura media": [_L0.loc[_L0.run_idx == r, "y"].mean() for r in _pyro]})
print(_tab.round(1).to_string(index=False))

print("\nMesmo modelo e mesmas previsoes, avaliadas com e sem os ensaios 51 e 52 (diagnostico, nao filtro):")
for _rot, _m in (("todas as leituras", np.ones(len(_L0), bool)),
                 ("sem os ensaios 51 e 52", ~_L0["run_idx"].isin([51, 52]).to_numpy())):
    _r = _mede(_L0["y"].to_numpy()[_m], _L0["pred"].to_numpy()[_m], _L0["componente"].to_numpy()[_m])
    print(f"  {_rot:24s} leituras {_m.sum():3d} | MAE {_r['MAE']:.3f} | RMSE {_r['RMSE']:.3f} | "
          f"R2 conjunto {_r['R2 conjunto']:.3f} | R2 dentro {_r['R2 dentro']:.3f}")

### Leitura

O coeficiente de determinação baixo do Oven HMI não decorre do modelo, e sim de dois ensaios. Os ensaios 48 a 52 são do tipo pyro e têm exatamente a mesma configuração declarada, mas os três primeiros medem entre 59 e 62 graus e os dois últimos, 51 e 52, cerca de 46 graus. Como as entradas são idênticas, nenhum modelo pode distingui-los, e a floresta prevê para os cinco o patamar dos três primeiros, errando os dois últimos em até 17 graus. As cinco maiores leituras respondem por 57 por cento do erro quadrático da região, e as dez maiores por 75 por cento. O coeficiente de determinação, que pesa o quadrado dos erros, é a métrica mais afetada.

Avaliadas sem esses dois ensaios, as mesmas previsões têm coeficiente de 0,43 e erro médio de 1,74 grau. O número é apenas diagnóstico. Os dois ensaios são medições reais, e retirá-los para elevar a métrica seria escolher o resultado; a pergunta sobre o que mudou neles foi encaminhada à parceira.

O coeficiente dentro do componente é negativo em todas as abordagens: dentro de cada componente, nenhum modelo explica a variação entre ensaios melhor que a média do próprio componente. A média histórica tem o maior coeficiente conjunto, 0,19, porque, ao prever o mesmo valor para todo ensaio pyro, erra menos nos dois ensaios frios; em troca, erra mais nos demais e tem erro médio maior que a floresta.

Nenhuma das abordagens alternativas supera a formulação entregue. Retirar o motivo piora o erro médio em cerca de 0,3 grau. A mescla com a regressão linear piora, porque os modelos lineares não captam a resposta da região. A mescla de floresta, gradiente e rede neural também piora, tanto no erro médio quanto no coeficiente de determinação. Coeficientes de 0,6 relatados por outros grupos só são comparáveis a estes se medidos com a mesma validação por configuração: numa divisão aleatória das leituras, o ensaio 51 fica no treino quando o 52 é previsto, e o modelo passa a acertar exatamente o caso que aqui não tem como ser acertado.